In [ ]:
# Setup: Download anonymous dataset
import os

# Create data directory if it doesn't exist
os.makedirs('./data', exist_ok=True)

# TODO: Add the wget command to download your synthetic dataset from an anonymous host
# !wget -q -O ./data/dataset.zip "https://anonymous.4open.science/api/repo/YOUR-REPO/file/dataset.zip"
# !unzip -q ./data/dataset.zip -d ./data/

In [ ]:
# a required dependency for vjepa
!pip install timm einops

In [ ]:
# need to clone repo to get code
!git clone https://github.com/facebookresearch/vjepa2.git

In [ ]:
# sys is a built-in python module
# we insert the path because we need to navigate to src in the vjepa repo
import sys
sys.path.insert(0, '/content/vjepa2')

In [ ]:
# downloading the actual checkpoint
import os
CKPT_PATH = '/content/vjepa2_1_vitb_dist_vitG_384.pt'
if not os.path.exists(CKPT_PATH):
    !wget https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt -O {CKPT_PATH}
print(f"Checkpoint exists: {os.path.exists(CKPT_PATH)}")
print(f"Checkpoint size: {os.path.getsize(CKPT_PATH) / 1e6:.1f} MB")

In [ ]:
import torch

In [ ]:
try:
    model = torch.hub.load(
        '/content/vjepa2',       # local path, not GitHub
        'vjepa2_1_vit_base_384',
        source='local',          # use local clone, skip download
        pretrained=False,        # don't try to download weights
    )
    print("Method A worked: loaded architecture from local hub")
    HUB_LOADED = True
except Exception as e:
    print(f"Method A failed: {e}")
    print("Falling back to Method B...")
    HUB_LOADED = False

In [ ]:
# loading cehckpoint weights
checkpoint = torch.load(CKPT_PATH, map_location='cpu', weights_only=True)

In [ ]:
# a map of weights and biases assigned to tensors
state_dict = checkpoint['encoder']
state_dict = {k.replace('module.', '').replace('backbone.', ''): v
              for k, v in state_dict.items()}

In [ ]:
print(type(model))
print(len(model))
model = model[0]

In [ ]:
# prior to this line, we had randomized weights. now we are using meta's trained weights
msg = model.load_state_dict(state_dict, strict=False)
print(f"\nLoad state dict message:")
print(f"  Missing keys:    {msg.missing_keys}")
print(f"  Unexpected keys: {msg.unexpected_keys}")

In [ ]:
# model.cuda() moves matrix mults, math operations to GPU, much faster
# model.eval() switches from training to inference mode
model = model.cuda().eval()
print(f"\nModel loaded on GPU. Param count: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# we're gonna print the transformer blocks (attention + MLP)

print("Model top level children: ")
for name, child in model.named_children():
    print(f"  {name}: {type(child).__name__}")

# finding the transformer blocks & printing them by checking attributes (blocks or encoder)
if hasattr(model, 'blocks'):
  print(f"\nFound model.blocks: {len(model.blocks)} transformer blocks")
  print(f"  Block 0 type: {type(model.blocks[0]).__name__}")
elif hasattr(model, 'encoder'):
  print(f"\nFound model.encoder (HuggingFace-style)")
else:
    print("\nWARNING: Couldn't find blocks attribute. Printing full model:")
    print(model)

In [ ]:
hooks = []
import numpy as np
for hook in hooks:
  hook.remove()
layer_features = {}

In [ ]:
# here we make "hooks", which capture output at each layer in vjepa even though it's usually lost
def make_hook(layer_name):
  def hook_fn(module, input, output):
    # expected output shape: batch, num_patches, embed_dim
    # for a vit-B:
      #   spatial patches = (384/16)^2 = 576
      #   temporal patches = 16/2 = 8
      #   total patches = 576 * 8 = 4608
    if (isinstance(output, tuple)):
      output = output[0]
    layer_features[layer_name] = output.detach().cpu()
  return hook_fn


In [ ]:

for i, block in enumerate(model.blocks):
    hook = block.register_forward_hook(make_hook(f"block_{i}"))
    hooks.append(hook)

In [ ]:
# determine what layer the blocks live in
if hasattr(model, 'blocks'):
  blocks = model.blocks
  block_attr = 'blocks'
elif hasattr(model, 'encoder') and hasattr(model.encoder, 'blocks'):
  blocks = model.encoder.blocks
  block_attr = 'encoder.blocks'
else:
    raise RuntimeError("Cannot find transformer blocks! Check model structure above.")

In [ ]:
# add hooks (output of each block)
for i, block in enumerate(blocks):
  hook = block.register_forward_hook(make_hook(f"block_{i}"))
  hooks.append(hook)

print(f"Registered {len(hooks)} hooks on {block_attr}")

In [ ]:
data = np.load('./elasticity_dataset6.0.npz')
print(list(data.keys()))
print({k: data[k].shape for k in data.keys()})

In [ ]:
VIDEO_PATH = './elasticity_dataset6.0.npz'

In [ ]:
# loading SINGULAR video
data = np.load(VIDEO_PATH)
print(f"Keys in .npz file: {list(data.keys())}")
frames = data['frames'][0]  # expected: (64, 224, 224, 3)
print(f"Raw frames shape: {frames.shape}, dtype: {frames.dtype}")

In [ ]:
import torch.nn.functional as F

In [ ]:
# we subsample 16 out of 64 frames with stride 4
frames_sub = frames[::4]
print(f"After stride-4 subsample: {frames_sub.shape}")

In [ ]:
# converting all pixel values to [0, 1] scale
frames_float = frames_sub.astype(np.float32) / 255.0

In [ ]:
# normalizing with ImageNet statistics (vjepa) is trained based on this normalization
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])
frames_norm = (frames_float - IMAGENET_MEAN) / IMAGENET_STD

In [ ]:
# rearrange (16, H, W, 3) -> (3, 16, H, W) again what vjepa is trained on
frames_chw = np.transpose(frames_norm, (3, 0, 1, 2));

In [ ]:
# converting to tensor and adding batch dimension (1)
video_tensor = torch.from_numpy(frames_chw).float().unsqueeze(0) # (1, 2, 16, 22, 4, 224)

In [ ]:
# resizing from 224x224 to 384x384
if video_tensor.shape[-1] == 224:
    # Resize spatial dims: (1, 3, 16, 224, 224) -> (1, 3, 16, 384, 384)
    B, C, T, H, W = video_tensor.shape
    # Reshape to (B*T, C, H, W) for F.interpolate, then reshape back
    video_flat = video_tensor.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)
    video_resized = F.interpolate(video_flat, size=(384, 384), mode='bilinear', align_corners=False)
    video_tensor = video_resized.reshape(B, T, C, 384, 384).permute(0, 2, 1, 3, 4)
    print(f"Resized 224 -> 384")

In [ ]:
video_tensor = video_tensor.cuda()
print(f"Final input tensor shape: {video_tensor.shape}")
print(f"  min={video_tensor.min().item():.3f}, max={video_tensor.max().item():.3f}, mean={video_tensor.mean().item():.3f}")

In [ ]:
# checking layer features:
#print(f"\n{'='*60}")
#print(f"Layer-wise feature shapes (from hooks)")
#print(f"{'='*60}")
#
#num_layers = len(layer_features)
#print(f"Total layers captured: {num_layers}")
#print()
#
#for name in sorted(layer_features.keys(), key = lambda x: int(x.split('_')[1])):
#  feat = layer_features[name]
#  print(f"{name}: shape={feat.shape}, "
#          f"min={feat.min():.3f}, max={feat.max():.3f}, "
#          f"mean={feat.mean():.3f}, std={feat.std():.3f}")

In [ ]:
with torch.no_grad():
    output = model(video_tensor)

In [ ]:
print(f"Total layers captured: {len(layer_features)}")

In [ ]:
print(f"\n{'='*60}")
print(f"Mean-pooled features (what you'll feed to Ridge probes)")
print(f"{'='*60}")

pooled_features = {}
for name in sorted(layer_features.keys(), key = lambda x: int(x.split('_')[1])):
  feat = layer_features[name] # (1, num_patches, embed_dim)
  pooled = feat.mean(dim = 1) # (1, embed_dim)
  pooled_features[name] = pooled
  print(f"{name}: pooled shape={pooled.shape}, "
          f"norm={torch.norm(pooled).item():.3f}")

In [ ]:
# checking for degenerate features (all 0s, NaN, identical across all layers)
print(f"\n--- Sanity Checks ---")

In [ ]:
# check no NaNs
has_nan = any(torch.isnan(v).any() for v in pooled_features.values())
print(f"Any NaN values: {has_nan} {'❌ BAD' if has_nan else '✅ OK'}")

In [ ]:
# check no features are all 0
all_zero = any(torch.all(v == 0) for v in pooled_features.values())
print(f"Any all-zero layers: {all_zero} {'❌ BAD' if all_zero else '✅ OK'}")

In [ ]:
# check that features vary across layers
first_key = sorted(pooled_features.keys(), key = lambda x: int(x.split('_')[1]))[0]
last_key = sorted(pooled_features.keys(), key = lambda x: int(x.split('_')[1]))[-1]
cos_sim = torch.nn.functional.cosine_similarity(
    pooled_features[first_key], pooled_features[last_key], dim = 1
).item()
print(f"Cosine sim (first vs last layer): {cos_sim:.4f} "
      f"{'⚠️ Suspiciously similar' if cos_sim > 0.99 else '✅ Features evolve across layers'}")

In [ ]:
# check feature magnitutes are within reasonable range
norms = [torch.norm(v).item() for v in pooled_features.values()]
print(f"Feature norm range: [{min(norms):.2f}, {max(norms):.2f}] "
      f"{'✅ OK' if max(norms) < 1000 else '⚠️ Unusually large'}")

In [ ]:
# cleaning up hooks (space concern)
for hook in hooks:
  hook.remove()
print(f"Removed {len(hooks)} hooks")

In [ ]:
# clear the GPU memory
del video_tensor, output, layer_features
torch.cuda.empty_cache()
print(f"GPU memory freed")

In [ ]:
num_layers = len(model.blocks)
num_layers

In [ ]:

print(f"""
{'='*60}
SUMMARY — Single Video Test
{'='*60}
Model:        V-JEPA 2.1 ViT-B/16 384
Checkpoint:   {CKPT_PATH.split('/')[-1]}
Video:        {VIDEO_PATH.split('/')[-1]}
Input shape:  (1, 3, 16, 384, 384)
Layers:       {num_layers} transformer blocks
Per-layer:    (1, num_patches, embed_dim)
Mean-pooled:  (1, embed_dim) per layer

Next steps:
  1. If shapes look right and sanity checks pass, you're good!
  2. Next: loop over all 2,000 videos, extract + save features.
  3. Stack into (500, {num_layers}, embed_dim) per property.
  4. Save to Drive as .npz files.
{'='*60}
""")

# Features for All Videos

In [ ]:
# setting up paths and settings
import os
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

In [ ]:
# unzipping data file
!unzip "./pymunk_video_data_6.0.zip" -d /content/data/
!ls /content/data/

In [ ]:
# paths from Drive
DATA_DIR = './data'
SAVE_DIR = './features6.0'
os.makedirs(SAVE_DIR, exist_ok = True)

In [ ]:
# 4 property datasets. each with 'frames' key with shape (500, 64, 224, 224, 3)
# and a labels key
PROPERTIES = {
    'elasticity': '/content/data/elasticity/dataset.npz',
    'friction':   '/content/data/friction/dataset.npz',
    'mass':       '/content/data/mass/dataset.npz',
    'drag':       '/content/data/drag/dataset.npz',
}

In [ ]:
# ImageNet normalization constants
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

In [ ]:
# printing what's in each file to verify key names
#for prop, path in PROPERTIES.items():
#    data = np.load(path, mmap_mode='r')
#    print(f"{prop}: keys={list(data.keys())}, ", end='')
#    for k in data.keys():
#        print(f"{k}.shape={data[k].shape}, ", end='')
#    print()

In [ ]:
# temp replacement for above block
for prop, path in PROPERTIES.items():
    print(f"{prop}: {path} — exists: {os.path.exists(path)}")

In [ ]:
# same logic as single-video test, but in a function
def preprocess_video(frames, target_size = 384):
  # Subsample 16 frames with stride 4
  frames_sub = frames[::4]  # (16, 224, 224, 3)

  # converting to flaot and normalizing
  frames_float = frames_sub.astype(np.float32) / 255.0
  frames_norm = (frames_float - IMAGENET_MEAN) / IMAGENET_STD

  # rearranging shape
  frames_chw = np.transpose(frames_norm, (3, 0, 1, 2))

  # add batch dimension to tensor
  tensor = torch.from_numpy(frames_chw).float().unsqueeze(0)  # (1, 3, 16, H, W)

  # resizing from 224x224 to 384x384
  if tensor.shape[-1] != target_size:
      B, C, T, H, W = tensor.shape
      flat = tensor.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)
      resized = F.interpolate(flat, size=(target_size, target_size),
                              mode='bilinear', align_corners=False)
      tensor = resized.reshape(B, T, C, target_size, target_size).permute(0, 2, 1, 3, 4)

  return tensor.cuda()

In [ ]:
NUM_LAYERS = len(model.blocks) # should be 12 for ViT-B
EMBED_DIM = None

In [ ]:
def extract_vjepa_features(model, video_tensor):
  # run one vid through vjepa with hooks, returning mean-pooled features at each layer
  # return type is np array of shape (num_layers, embed_dim)

  layer_outputs = {}
  hooks = []

  def make_hook(layer_name):
    def hook_fn(module, input, output):
      if (isinstance(output, tuple)):
        output = output[0]
      layer_outputs[layer_name] = output.detach().cpu()
    return hook_fn

  # registering the hooks
  for i, block in enumerate(model.blocks):
    hooks.append(block.register_forward_hook(make_hook(f"block_{i}")))

  # forward bass
  with torch.no_grad():
    model(video_tensor)

  # we're immediately removing hooks
  for h in hooks:
    h.remove()

  # mean-pool across patches: (1, num_patches, embed_dim) -> (embed_dim,)
  features = []
  for i in range(NUM_LAYERS):
    feat = layer_outputs[f"block_{i}"]  # (1, num_patches, embed_dim)
    pooled = feat.mean(dim = 1).squeeze(0).numpy() # (embed_dim,)

    features.append(pooled)

  return np.stack(features) # (num_layers, embed_dim)

In [ ]:
torch.manual_seed(0)

In [ ]:
import zipfile, io

def load_single_video_from_npz(npz_path, video_index=0):
    """Load one video from a compressed .npz without decompressing the whole thing."""
    video_bytes = 64 * 224 * 224 * 3  # 9,633,792 bytes

    with zipfile.ZipFile(npz_path, 'r') as zf:
        with zf.open('frames.npy') as f:
            # Parse .npy header
            magic = f.read(6)
            version = f.read(2)
            if version == b'\x02\x00':
                header_len = int.from_bytes(f.read(4), 'little')
            else:
                header_len = int.from_bytes(f.read(2), 'little')
            f.read(header_len)

            # Skip to the desired video
            f.read(video_bytes * video_index)
            raw = f.read(video_bytes)

        return np.frombuffer(raw, dtype=np.uint8).reshape(64, 224, 224, 3)

# Test
with torch.no_grad():
    test_frames = load_single_video_from_npz(list(PROPERTIES.values())[0], 0)
    test_feat = extract_vjepa_features(model, preprocess_video(test_frames))
print(f"V-JEPA feature shape per video: {test_feat.shape}")
EMBED_DIM = test_feat.shape[1]
del test_frames, test_feat
torch.cuda.empty_cache()

In [ ]:
# quick test to detect embed_dim
with torch.no_grad():
    test_frames = load_single_video_from_npz(list(PROPERTIES.values())[0], 0)
    test_feat = extract_vjepa_features(model, preprocess_video(test_frames))
    del test_frames
    torch.cuda.empty_cache()

print(f"V-JEPA feature shape per video: {test_feat.shape}")  # expect (12, 768)
EMBED_DIM = test_feat.shape[1]

In [ ]:
import zipfile, io

def load_labels_from_npz(npz_path):
    """Load just the labels array from an .npz."""
    with zipfile.ZipFile(npz_path, 'r') as zf:
        # Find the non-frames key
        names = [n.replace('.npy', '') for n in zf.namelist()]
        LABEL_KEYS = {'elasticity_values', 'friction_values', 'mass_ratio_m', 'drag_coefficients'}
        labels_key = [n for n in names if n in LABEL_KEYS][0]
        with zf.open(f'{labels_key}.npy') as f:
            return labels_key, np.load(io.BytesIO(f.read()))

def stream_videos_from_npz(npz_path, num_videos=500):
    """Yield one video at a time from a compressed .npz without loading all into RAM."""
    video_bytes = 64 * 224 * 224 * 3

    with zipfile.ZipFile(npz_path, 'r') as zf:
        with zf.open('frames.npy') as f:
            # Parse .npy header
            magic = f.read(6)
            version = f.read(2)
            if version == b'\x02\x00':
                header_len = int.from_bytes(f.read(4), 'little')
            else:
                header_len = int.from_bytes(f.read(2), 'little')
            f.read(header_len)

            for i in range(num_videos):
                raw = f.read(video_bytes)
                if len(raw) < video_bytes:
                    print(f"Warning: truncated at video {i}")
                    break
                yield np.frombuffer(raw, dtype=np.uint8).reshape(64, 224, 224, 3).copy()

for prop, path in PROPERTIES.items():
    save_path = os.path.join(SAVE_DIR, f'vjepa_{prop}.npz')

    if os.path.exists(save_path):
        print(f"[SKIP] {prop} — already exists at {save_path}")
        continue

    print(f"\n{'='*50}")
    print(f"Extracting V-JEPA features for: {prop}")
    print(f"{'='*50}")

    labels_key, labels = load_labels_from_npz(path)
    print(f"  Labels key: '{labels_key}', shape: {labels.shape}")

    # Also load scenario_id for scenario-level splits
    with zipfile.ZipFile(path, 'r') as zf:
        with zf.open('scenario_id.npy') as f:
            scenario_ids = np.load(io.BytesIO(f.read()))
    print(f"  Scenario IDs shape: {scenario_ids.shape}, unique: {len(np.unique(scenario_ids))}")

    num_videos = labels.shape[0]
    all_features = np.zeros((num_videos, NUM_LAYERS, EMBED_DIM), dtype=np.float32)

    for i, frames in enumerate(tqdm(stream_videos_from_npz(path, num_videos),
                                     total=num_videos, desc=prop)):
        video_tensor = preprocess_video(frames)
        all_features[i] = extract_vjepa_features(model, video_tensor)
        del video_tensor, frames
        torch.cuda.empty_cache()

    np.savez_compressed(save_path,
                        features=all_features,
                        labels=labels,
                        scenario_ids=scenario_ids,
                        property_name=prop)
    print(f"  Saved: {save_path} ({os.path.getsize(save_path) / 1e6:.1f} MB)")

print("\nV-JEPA extraction complete")

In [ ]:
# === Temporal-Integration-Length: V-JEPA k-Frame Extraction ===
# For each property, extract V-JEPA features under k ∈ {2, 4, 8, 16} where
# k = number of DISTINCT frames in V-JEPA's 16-slot input. Frames are
# evenly spaced across the source 64-frame video; each distinct frame is
# block-duplicated to fill 16/k contiguous slots. k=16 is reused from the
# headline pipeline (vjepa_{prop}.npz already on disk).
#
# Tests how V-JEPA's per-property representation degrades as temporal
# information is starved. Steep degradation = property requires temporal
# integration; flat = property already encoded in spatial signal.

import io, os, zipfile
import numpy as np
import torch
from tqdm import tqdm

K_VALUES = [2, 4, 8]                # k=16 is the existing vjepa_{prop}.npz
SAVE_DIR_KF = "./features6.0_kframes"
os.makedirs(SAVE_DIR_KF, exist_ok=True)

# Source videos are 64 frames; preprocess_video does frames[::4] -> 16 frames.
# To get k distinct frames in those 16 slots, we choose k source indices that
# are evenly spaced AND lie on the [::4] grid, then build a fake 64-frame video
# where the right blocks of source positions hold each chosen frame.
SOURCE_LEN = 64
INPUT_LEN = 16  # what preprocess_video produces after [::4]


def build_kframe_video(frames, k):
    """
    Construct a fake 64-frame video so that after preprocess_video's [::4]
    subsampling, the resulting 16-frame input contains k distinct frames,
    each block-duplicated (16/k) times in contiguous slots.

    Example k=4: input slots [0..3]=f0, [4..7]=f21, [8..11]=f42, [12..15]=f63.
    The chosen source frames are evenly spaced across the full 64-frame span
    so temporal extent is held constant; only resolution varies with k.
    """
    assert INPUT_LEN % k == 0, f"k={k} must divide {INPUT_LEN}"
    block = INPUT_LEN // k                           # how many slots per distinct frame
    # k evenly-spaced source indices spanning [0, SOURCE_LEN-1]
    src_idx = np.linspace(0, SOURCE_LEN - 1, k).round().astype(int)
    # Build the fake 64-frame video. For each of the 16 input slots i, that slot
    # comes from source position 4*i (because preprocess does [::4]). We want
    # input slot i to hold chosen_frame[i // block].
    fake = np.empty_like(frames)                     # (64, 224, 224, 3)
    # Fill all 64 source positions; only positions 0,4,8,...,60 will survive [::4],
    # so we just need those to be correct, but filling the rest with the same
    # block keeps the structure obvious and costs nothing.
    for input_slot in range(INPUT_LEN):
        which_distinct = input_slot // block          # 0..k-1
        chosen = frames[src_idx[which_distinct]]      # (224, 224, 3)
        # Source positions [4*input_slot, 4*input_slot+4) all map to this slot
        # after [::4] (only 4*input_slot survives, but fill the window anyway).
        fake[4*input_slot : 4*input_slot + 4] = chosen
    return fake


# --- Sanity check the construction on a tiny synthetic case ----------------
# Verify that for k=4, the post-[::4] input has 4 distinct frames each
# repeated 4 times in contiguous blocks. Catches off-by-one bugs early.
_test = np.arange(SOURCE_LEN, dtype=np.uint8).reshape(SOURCE_LEN, 1, 1, 1)
_test = np.broadcast_to(_test, (SOURCE_LEN, 224, 224, 3)).copy()
_built = build_kframe_video(_test, k=4)
_after_subsample = _built[::4][:, 0, 0, 0]            # the 16 input slots' frame IDs
_expected_distinct = 4
_expected_block = 4
_distinct_count = len(set(_after_subsample.tolist()))
assert _distinct_count == _expected_distinct, \
    f"k=4 sanity check failed: got {_distinct_count} distinct frames, expected {_expected_distinct}"
# Each distinct frame should occupy exactly 4 contiguous slots
for b in range(_expected_distinct):
    block_slice = _after_subsample[b*_expected_block : (b+1)*_expected_block]
    assert len(set(block_slice.tolist())) == 1, f"k=4 block {b} not uniform: {block_slice}"
print(f"[sanity] k=4 construction OK — slots: {_after_subsample.tolist()}")
del _test, _built, _after_subsample


# --- Main extraction loop --------------------------------------------------
for k in K_VALUES:
    print(f"\n{'='*60}\n  k = {k} distinct frames (block size = {INPUT_LEN // k})\n{'='*60}")

    for prop, path in PROPERTIES.items():
        save_path = os.path.join(SAVE_DIR_KF, f"vjepa_k{k}_{prop}.npz")
        if os.path.exists(save_path):
            print(f"[SKIP] {prop} k={k} — already exists at {save_path}")
            continue

        labels_key, labels = load_labels_from_npz(path)
        with zipfile.ZipFile(path, 'r') as zf:
            with zf.open('scenario_id.npy') as f:
                scenario_ids = np.load(io.BytesIO(f.read()))

        N = labels.shape[0]
        # We only care about the per-property peak layer for probing, but we save
        # all 12 layers anyway — costs little extra disk and lets you re-probe
        # at any layer later without re-extracting features.
        all_features = np.empty((N, NUM_LAYERS, EMBED_DIM), dtype=np.float32)

        for vid_idx, frames in enumerate(tqdm(
            stream_videos_from_npz(path, N), desc=f"{prop} k={k}", total=N
        )):
            fake_video = build_kframe_video(frames, k)        # (64, 224, 224, 3)
            video_tensor = preprocess_video(fake_video)        # (1, 3, 16, 384, 384) on CUDA
            all_features[vid_idx] = extract_vjepa_features(model, video_tensor)
            del video_tensor, fake_video, frames
            torch.cuda.empty_cache()

        np.savez_compressed(
            save_path,
            features=all_features,
            labels=labels,
            scenario_ids=scenario_ids,
            property_name=prop,
            k_distinct_frames=k,
            block_size=INPUT_LEN // k,
            duplication_scheme="block_contiguous",
        )
        print(f"  saved {save_path}  shape {all_features.shape}  "
              f"({os.path.getsize(save_path) / 1e6:.1f} MB)")

print("\nk-frame extraction complete.")
print(f"k=16 condition: reuse existing vjepa_{{prop}}.npz from features6.0/")

In [ ]:
# === Single-Frame-Duplicated Control: V-JEPA ===
# Extract V-JEPA features where input is the LAST frame duplicated 16 times.
# Tests whether peak-layer probing can recover property from one fully-integrated
# frame. If R² ≈ 0, temporal-integration confound is eliminated.

import io, zipfile, os
import numpy as np
import torch
from tqdm import tqdm

SAVE_DIR_SF = "./features6.0_singleframe"
os.makedirs(SAVE_DIR_SF, exist_ok=True)

for prop, path in PROPERTIES.items():
    save_path = f"{SAVE_DIR_SF}/vjepa_lastframe_{prop}.npz"
    if os.path.exists(save_path):
        print(f"[skip] {save_path} exists")
        continue

    print(f"\n=== {prop.upper()} (last-frame-duplicated) ===")

    labels_key, labels = load_labels_from_npz(path)
    with zipfile.ZipFile(path, 'r') as zf:
        with zf.open('scenario_id.npy') as f:
            scenario_ids = np.load(io.BytesIO(f.read()))

    N = labels.shape[0]
    all_features = np.empty((N, NUM_LAYERS, EMBED_DIM), dtype=np.float32)

    for vid_idx, frames in enumerate(tqdm(
        stream_videos_from_npz(path, N), desc=f"{prop} last-frame", total=N
    )):
        # Build a fake 64-frame video where every frame is the last frame.
        # preprocess_video subsamples [::4] giving 16 identical frames, then normalizes/resizes.
        last_frame = frames[-1]                                   # (224, 224, 3)
        fake_video = np.repeat(last_frame[None, ...], 64, axis=0) # (64, 224, 224, 3)

        video_tensor = preprocess_video(fake_video)               # (1, 3, 16, 384, 384) on CUDA
        all_features[vid_idx] = extract_vjepa_features(model, video_tensor)

        del video_tensor, fake_video
        torch.cuda.empty_cache()

    np.savez_compressed(
        save_path,
        features=all_features,
        labels=labels,
        scenario_ids=scenario_ids,
        property_name=prop,
        control="last_frame_duplicated_x16",
    )
    print(f"  saved {save_path}, shape {all_features.shape}")

In [ ]:
# installing CLIP + DINOv2 dependencies
!pip install open_clip_torch

In [ ]:
# loading CLIP model
import open_clip

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-16', pretrained='openai'
)
clip_model = clip_model.visual # only need the vision encoder
clip_model = clip_model.cuda().eval()
print(f"CLIP ViT-B/16 loaded. Params: {sum(p.numel() for p in clip_model.parameters()) / 1e6:.1f}M")

In [ ]:
# CLIP feature extraction
# it's an image model, so we process each of 16 frames and average across frames

def preprocess_for_clip(frames, target_size = 224):
  # CLIP wants (B, 3, 224, 224) per image
  # we have to preprocess images

  frames_sub = frames[::4] # (16, 224, 224, 3)
  frames_float = frames_sub.astype(np.float32) / 255.0

  # CLIP normalization (similar to ImageNet)
  clip_mean = np.array([0.48145466, 0.4578275, 0.40821073])
  clip_std  = np.array([0.26862954, 0.26130258, 0.27577711])

  frames_norm = (frames_float - clip_mean) / clip_std

  # (16, 224, 224, 3) -> (16, 3, 224, 224)
  frames_chw = np.transpose(frames_norm, (0, 3, 1, 2))
  tensor = torch.from_numpy(frames_chw).float()

  # CLIP wants 224 x 224, so no need to resize

  return tensor.cuda()


In [ ]:
def extract_clip_features(clip_model, frames):
  # extract CLIP features for one video
  # process 16 frames independently, and then average them

  frame_tensors = preprocess_for_clip(frames) # (16, 3, 224, 224)

  with torch.no_grad():
    # process all 16 frames
    frame_features = clip_model(frame_tensors)

  video_feature = frame_features.mean(dim = 0).cpu().numpy()
  return video_feature

In [ ]:
test_clip = extract_clip_features(clip_model,
    load_single_video_from_npz(list(PROPERTIES.values())[0], 0))
print(f"CLIP feature shape per video: {test_clip.shape}")
CLIP_DIM = test_clip.shape[0]

In [ ]:
# === Per-Layer CLIP Feature Extraction (with hooks) ===
import io, zipfile, os
import numpy as np
import torch
from tqdm import tqdm

CLIP_NUM_LAYERS = 12
CLIP_EMBED_DIM = 768   # ViT-B/16 visual transformer hidden size (pre-projection)

# --- Set up hooks on every CLIP visual transformer block ---
# Note: cell 60 reassigned clip_model = clip_model.visual, so resblocks live at
# clip_model.transformer.resblocks (not clip_model.visual.transformer.resblocks).
clip_layer_features = {}
clip_hooks = []

def make_clip_hook(name):
    def hook(module, inp, out):
        clip_layer_features[name] = out
    return hook

resblocks = clip_model.transformer.resblocks
for i, block in enumerate(resblocks):
    h = block.register_forward_hook(make_clip_hook(f"clip_block_{i}"))
    clip_hooks.append(h)
print(f"Registered {len(clip_hooks)} hooks on CLIP visual blocks")

# --- Extract per-layer features ---
for prop, path in PROPERTIES.items():
    save_path = os.path.join(SAVE_DIR, f"clip_perlayer_{prop}.npz")
    if os.path.exists(save_path):
        print(f"[skip] {save_path} exists")
        continue

    print(f"\n=== CLIP per-layer: {prop.upper()} ===")
    labels_key, labels = load_labels_from_npz(path)
    with zipfile.ZipFile(path, 'r') as zf:
        with zf.open('scenario_id.npy') as f:
            scenario_ids = np.load(io.BytesIO(f.read()))

    N = labels.shape[0]
    all_features = np.empty((N, CLIP_NUM_LAYERS, CLIP_EMBED_DIM), dtype=np.float32)

    for vid_idx, frames in enumerate(tqdm(
        stream_videos_from_npz(path, N), desc=f"CLIP {prop}", total=N
    )):
        frame_tensors = preprocess_for_clip(frames)   # (16, 3, 224, 224) on CUDA

        clip_layer_features.clear()
        with torch.no_grad():
            _ = clip_model(frame_tensors)

        for layer_i in range(CLIP_NUM_LAYERS):
            feat = clip_layer_features[f"clip_block_{layer_i}"]
            # CLIP resblocks: (seq_len, batch=16, dim) — seq_first
            feat = feat.permute(1, 0, 2)             # (16, seq_len, dim)
            patch_feat = feat[:, 1:, :]              # drop CLS
            pooled_per_frame = patch_feat.mean(dim=1)
            pooled_video = pooled_per_frame.mean(dim=0)
            all_features[vid_idx, layer_i] = pooled_video.cpu().numpy()

    np.savez_compressed(
        save_path,
        features=all_features,
        labels=labels,
        scenario_ids=scenario_ids,
        property_name=prop,
    )
    print(f"  saved {save_path}, shape {all_features.shape}")

for h in clip_hooks:
    h.remove()
clip_hooks = []

In [ ]:
# now we extract CLIP for ALL videos
for prop, path in PROPERTIES.items():
    save_path = os.path.join(SAVE_DIR, f'clip_{prop}.npz')

    if os.path.exists(save_path):
        print(f"[SKIP] {prop} — already exists")
        continue

    print(f"Extracting CLIP features for: {prop}")

    labels_key, labels = load_labels_from_npz(path)


    with zipfile.ZipFile(path, 'r') as zf:
        with zf.open('scenario_id.npy') as f:
            scenario_ids = np.load(io.BytesIO(f.read()))
    print(f"  Scenario IDs shape: {scenario_ids.shape}, unique: {len(np.unique(scenario_ids))}")


    num_videos = labels.shape[0]
    all_features = np.zeros((num_videos, CLIP_DIM), dtype=np.float32)

    for i, frames in enumerate(tqdm(stream_videos_from_npz(path, num_videos),
                                     total=num_videos, desc=prop)):
        all_features[i] = extract_clip_features(clip_model, frames)
        del frames

    np.savez_compressed(save_path,
                        features=all_features,
                        labels=labels,
                        scenario_ids=scenario_ids,
                        property_name=prop)
    print(f"  Saved: {save_path} ({os.path.getsize(save_path) / 1e6:.1f} MB)")

print("\nCLIP extraction complete")

In [ ]:
# freeing CLIP from the GPU
del clip_model
torch.cuda.empty_cache()

In [ ]:
# loading DINOv2 model
dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
dino_model = dino_model.cuda().eval()
print(f"DINOv2 ViT-B/14 loaded. Params: {sum(p.numel() for p in dino_model.parameters()) / 1e6:.1f}M")

In [ ]:
# DINOv2 is also an image model, so we process frames independently and then average
# DINOv2 wants 518x518, or a multiple of 14

def preprocess_for_dino(frames):
  # ImageNet normalization and reordering tensor
  frames_sub = frames[::4]  # (16, 224, 224, 3)
  frames_float = frames_sub.astype(np.float32) / 255.0
  frames_norm = (frames_float - IMAGENET_MEAN) / IMAGENET_STD
  frames_chw = np.transpose(frames_norm, (0, 3, 1, 2))  # (16, 3, 224, 224)
  return torch.from_numpy(frames_chw).float().cuda()

In [ ]:
# extracting DINOv2 features for one video
def extract_dino_features(dino_model, frames):
  frame_tensors = preprocess_for_dino(frames)  # (16, 3, 224, 224)

  with torch.no_grad():
      # DINOv2 forward returns CLS token by default
      frame_features = dino_model(frame_tensors)  # (16, 768)

  video_feature = frame_features.mean(dim=0).cpu().numpy()  # (768,)
  return video_feature

In [ ]:
# testing the feature shapes
test_dino = extract_dino_features(dino_model,
    load_single_video_from_npz(list(PROPERTIES.values())[0], 0))
print(f"DINOv2 feature shape per video: {test_dino.shape}")
DINO_DIM = test_dino.shape[0]

In [ ]:
# === Per-Layer DINOv2 Feature Extraction (with hooks) ===
import io, zipfile, os
import numpy as np
import torch
from tqdm import tqdm

DINO_NUM_LAYERS = 12
DINO_EMBED_DIM = 768

# --- Set up hooks on every DINOv2 transformer block ---
dino_layer_features = {}
dino_hooks = []

def make_dino_hook(name):
    def hook(module, inp, out):
        dino_layer_features[name] = out
    return hook

dino_blocks = dino_model.blocks
for i, block in enumerate(dino_blocks):
    h = block.register_forward_hook(make_dino_hook(f"dino_block_{i}"))
    dino_hooks.append(h)
print(f"Registered {len(dino_hooks)} hooks on DINOv2 blocks")

# DINOv2 ViT-B/14 standard variant has 1 CLS token and no registers.
# Set to 5 if you loaded the with-registers variant.
NUM_PREFIX_TOKENS = 1

# --- Extract per-layer features ---
for prop, path in PROPERTIES.items():
    save_path = os.path.join(SAVE_DIR, f"dino_perlayer_{prop}.npz")
    if os.path.exists(save_path):
        print(f"[skip] {save_path} exists")
        continue

    print(f"\n=== DINOv2 per-layer: {prop.upper()} ===")
    labels_key, labels = load_labels_from_npz(path)
    with zipfile.ZipFile(path, 'r') as zf:
        with zf.open('scenario_id.npy') as f:
            scenario_ids = np.load(io.BytesIO(f.read()))

    N = labels.shape[0]
    all_features = np.empty((N, DINO_NUM_LAYERS, DINO_EMBED_DIM), dtype=np.float32)

    for vid_idx, frames in enumerate(tqdm(
        stream_videos_from_npz(path, N), desc=f"DINO {prop}", total=N
    )):
        frame_tensors = preprocess_for_dino(frames)  # (16, 3, 224, 224) on CUDA

        dino_layer_features.clear()
        with torch.no_grad():
            _ = dino_model(frame_tensors)

        for layer_i in range(DINO_NUM_LAYERS):
            feat = dino_layer_features[f"dino_block_{layer_i}"]   # (16, seq_len, 768)
            patch_feat = feat[:, NUM_PREFIX_TOKENS:, :]
            pooled_per_frame = patch_feat.mean(dim=1)
            pooled_video = pooled_per_frame.mean(dim=0)
            all_features[vid_idx, layer_i] = pooled_video.cpu().numpy()

    np.savez_compressed(
        save_path,
        features=all_features,
        labels=labels,
        scenario_ids=scenario_ids,
        property_name=prop,
    )
    print(f"  saved {save_path}, shape {all_features.shape}")

for h in dino_hooks:
    h.remove()
dino_hooks = []

In [ ]:
# now we extract DINOv2 features for all videos

for prop, path in PROPERTIES.items():
    save_path = os.path.join(SAVE_DIR, f'dino_{prop}.npz')

    if os.path.exists(save_path):
        print(f"[SKIP] {prop} — already exists")
        continue

    print(f"Extracting DINOv2 features for: {prop}")

    labels_key, labels = load_labels_from_npz(path)


    # Also load scenario_id for scenario-level splits
    with zipfile.ZipFile(path, 'r') as zf:
        with zf.open('scenario_id.npy') as f:
            scenario_ids = np.load(io.BytesIO(f.read()))
    print(f"  Scenario IDs shape: {scenario_ids.shape}, unique: {len(np.unique(scenario_ids))}")


    num_videos = labels.shape[0]
    all_features = np.zeros((num_videos, DINO_DIM), dtype=np.float32)

    for i, frames in enumerate(tqdm(stream_videos_from_npz(path, num_videos),
                                     total=num_videos, desc=prop)):
        all_features[i] = extract_dino_features(dino_model, frames)
        del frames

    np.savez_compressed(save_path,
                        features=all_features,
                        labels=labels,
                        scenario_ids=scenario_ids,
                        property_name=prop)
    print(f"  Saved: {save_path} ({os.path.getsize(save_path) / 1e6:.1f} MB)")

print("\nDINOv2 extraction complete")

In [ ]:
# empty the cache
del dino_model
torch.cuda.empty_cache()

In [ ]:
# we verify everything worked
print(f"\n{'='*60}")
print(f"All saved feature files:")
print(f"{'='*60}")
total_size = 0
for f in sorted(os.listdir(SAVE_DIR)):
    if f.endswith('.npz'):
        fpath = os.path.join(SAVE_DIR, f)
        size = os.path.getsize(fpath) / 1e6
        total_size += size

        data = np.load(fpath)
        print(f"  {f:30s} | {size:6.1f} MB | features={data['features'].shape}, labels={data['labels'].shape}")

print(f"\nTotal: {total_size:.1f} MB")
print(f"\n✅ All done! Download the '{SAVE_DIR}' folder to your laptop for probing.")


In [ ]:
# sanity checks

import numpy as np

data = np.load('./features6.0/vjepa_friction.npz')
feats = data['features']  # (500, 12, 768)
labels = data['labels']

# 1. Basic stats — are values in a reasonable range?
print(f"Shape: {feats.shape}, dtype: {feats.dtype}")
print(f"Mean: {feats.mean():.4f}, Std: {feats.std():.4f}")
print(f"Min: {feats.min():.4f}, Max: {feats.max():.4f}")
print(f"Any NaN: {np.isnan(feats).any()}, Any Inf: {np.isinf(feats).any()}")

# 2. Are features different across videos? (if std~0, extraction was broken)
print(f"Std across videos (layer 0): {feats[:, 0, :].std(axis=0).mean():.4f}")
print(f"Std across videos (layer 11): {feats[:, 11, :].std(axis=0).mean():.4f}")

# 3. Are features different across layers? (if all layers identical, hooks were stale)
for i in range(11):
    cos = np.dot(feats[0, i], feats[0, i+1]) / (
        np.linalg.norm(feats[0, i]) * np.linalg.norm(feats[0, i+1]))
    print(f"  Layer {i} vs {i+1} cosine sim: {cos:.4f}")
# Should see <1.0, generally decreasing in later layers

# 4. Do features correlate at all with labels? (quick Pearson on one dimension)
from scipy.stats import pearsonr
# Pick the highest-correlated feature dimension at the last layer
correlations = [pearsonr(feats[:, -1, d], labels)[0] for d in range(768)]
print(f"Max absolute Pearson r (layer 11): {max(correlations, key=abs):.4f}")
# If this is >0.1 for at least some dimensions, there's signal

Probing script generated by Claude

In [ ]:
#!/usr/bin/env python3
"""
Ridge Regression Probing: V-JEPA 2 Frozen Encoder → Material Properties
========================================================================
Per-layer probing for V-JEPA, CLIP, DINOv2 + V-JEPA shuffled-frame control.
"""

import json
import numpy as np
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit

PROPERTIES = ["elasticity", "friction", "drag"]
N_LAYERS = 12
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS = np.logspace(-3, 6, 50)

DATA_DIR = Path("./features6.0")
OUT_DIR = Path("./probing_results6.0")


def load_features(prefix, prop):
    path = DATA_DIR / f"{prefix}_{prop}.npz"
    data = np.load(path)
    return data["features"], data["labels"], data["scenario_ids"]


def probe_single(X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_te = scaler.transform(X_test)
    model = RidgeCV(alphas=ALPHAS, scoring="r2")
    model.fit(X_tr, y_train)
    return r2_score(y_test, model.predict(X_te)), float(model.alpha_)


def probe_per_layer(feats, labels, train_idx, test_idx, label):
    r2s, alphas = [], []
    for layer in range(N_LAYERS):
        X = feats[:, layer, :]
        r2, alpha = probe_single(X[train_idx], X[test_idx],
                                 labels[train_idx], labels[test_idx])
        r2s.append(r2)
        alphas.append(alpha)
        print(f"  {label} Layer {layer+1:2d}  →  R² = {r2:.4f}  (α = {alpha:.2e})")
    return r2s, alphas


def main():
    results = {}

    for prop in PROPERTIES:
        print(f"\n{'='*60}\n  Property: {prop.upper()}\n{'='*60}")

        # V-JEPA
        v_feats, v_labels, v_scen = load_features("vjepa", prop)
        gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        tr, te = next(gss.split(v_feats, v_labels, groups=v_scen))
        assert len(set(v_scen[tr]) & set(v_scen[te])) == 0
        print(f"  Scenario split: {len(np.unique(v_scen[tr]))} train / {len(np.unique(v_scen[te]))} test\n")
        v_r2s, v_alphas = probe_per_layer(v_feats, v_labels, tr, te, "V-JEPA")

        # CLIP per-layer
        c_feats, c_labels, c_scen = load_features("clip_perlayer", prop)
        c_tr, c_te = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
                          .split(c_feats, c_labels, groups=c_scen))
        print(f"\n  --- CLIP per-layer ---")
        c_r2s, c_alphas = probe_per_layer(c_feats, c_labels, c_tr, c_te, "CLIP")

        # DINOv2 per-layer
        d_feats, d_labels, d_scen = load_features("dino_perlayer", prop)
        d_tr, d_te = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
                          .split(d_feats, d_labels, groups=d_scen))
        print(f"\n  --- DINOv2 per-layer ---")
        d_r2s, d_alphas = probe_per_layer(d_feats, d_labels, d_tr, d_te, "DINOv2")

        # V-JEPA shuffled-frame control
        s_feats, s_labels, s_scen = load_features("shuffled_frame_vjepa", prop)
        s_tr, s_te = next(GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
                          .split(s_feats, s_labels, groups=s_scen))
        print(f"\n  --- Shuffled-Frame Control (V-JEPA) ---")
        s_r2s, s_alphas = probe_per_layer(s_feats, s_labels, s_tr, s_te, "Shuf")

        peak = int(np.argmax(v_r2s))
        c_best = int(np.argmax(c_r2s))
        d_best = int(np.argmax(d_r2s))
        print(f"\n  ★ V-JEPA Peak: Layer {peak+1} (R² = {v_r2s[peak]:.4f})")
        print(f"    CLIP best:   L{c_best+1} (R² = {c_r2s[c_best]:.4f})  Δ = {v_r2s[peak] - c_r2s[c_best]:+.4f}")
        print(f"    DINOv2 best: L{d_best+1} (R² = {d_r2s[d_best]:.4f})  Δ = {v_r2s[peak] - d_r2s[d_best]:+.4f}")
        print(f"    vs Shuf @ peak: Δ = {v_r2s[peak] - s_r2s[peak]:+.4f}")

        results[prop] = {
            "vjepa_r2": v_r2s, "vjepa_alphas": v_alphas,
            "clip_r2": c_r2s, "clip_alphas": c_alphas,
            "clip_best_layer": c_best + 1, "clip_best_r2": c_r2s[c_best],
            "dino_r2": d_r2s, "dino_alphas": d_alphas,
            "dino_best_layer": d_best + 1, "dino_best_r2": d_r2s[d_best],
            "peak_layer": peak + 1, "peak_r2": v_r2s[peak],
            "vjepa_shuffled_r2": s_r2s, "vjepa_shuffled_alphas": s_alphas,
            "vjepa_shuffled_peak_layer_r2": s_r2s[peak],
        }

    print(f"\n{'='*60}\n  SUMMARY\n{'='*60}")
    for prop in PROPERTIES:
        r = results[prop]
        print(f"  {prop:12s}  V-JEPA L{r['peak_layer']:2d} R²={r['peak_r2']:.4f}  |  "
              f"CLIP best L{r['clip_best_layer']:2d} R²={r['clip_best_r2']:.4f}  |  "
              f"DINO best L{r['dino_best_layer']:2d} R²={r['dino_best_r2']:.4f}  |  "
              f"shuf@peak={r['vjepa_shuffled_peak_layer_r2']:.4f}")

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(OUT_DIR / "probe_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nSaved → {OUT_DIR / 'probe_results.json'}")


if __name__ == "__main__":
    main()

In [ ]:
# === Temporal-Integration-Length: Probe at Each Property's Peak Layer ===
# For each k ∈ {2, 4, 8, 16}, fit a ridge probe at the property's headline-result
# peak layer and record R². k=16 reuses the existing vjepa_{prop}.npz features
# from features6.0/. Output: how R² degrades with temporal starvation.

import json
import numpy as np
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit

# Peak layers from the headline V-JEPA results (1-indexed in paper, 0-indexed here).
# Source: project doc — elasticity L8, friction L12, drag L11.
PEAK_LAYERS = {"elasticity": 7, "friction": 11, "drag": 10}
K_VALUES = [2, 4, 8, 16]
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS = np.logspace(-3, 6, 50)

DATA_DIR_K = Path("./features6.0_kframes")
DATA_DIR_HEADLINE = Path("./features6.0")
OUT_DIR = Path("./probing_results6.0")


def load_kframe_features(prop, k):
    """k=16 lives in the headline directory (existing vjepa_{prop}.npz);
    k<16 lives in features6.0_kframes/ with the k embedded in the filename."""
    if k == 16:
        path = DATA_DIR_HEADLINE / f"vjepa_{prop}.npz"
    else:
        path = DATA_DIR_K / f"vjepa_k{k}_{prop}.npz"
    data = np.load(path)
    return data["features"], data["labels"], data["scenario_ids"]


def probe_at_layer(feats, labels, scen, layer):
    """Same probing recipe as the headline pipeline: scenario-level split with
    random_state=42, standardize on train, RidgeCV with the same alpha grid."""
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    tr, te = next(gss.split(feats, labels, groups=scen))
    X_tr = feats[tr, layer, :]
    X_te = feats[te, layer, :]
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    model = RidgeCV(alphas=ALPHAS, scoring="r2").fit(X_tr_s, labels[tr])
    r2 = r2_score(labels[te], model.predict(X_te_s))
    return float(r2), float(model.alpha_)


# --- Run -------------------------------------------------------------------
print(f"{'='*60}\n  Temporal-integration-length probing\n{'='*60}")
results = {}
for prop in ["elasticity", "friction", "drag"]:
    L = PEAK_LAYERS[prop]
    print(f"\n--- {prop.upper()}  (peak layer L{L+1}) ---")
    per_k = {}
    for k in K_VALUES:
        feats, labels, scen = load_kframe_features(prop, k)
        r2, alpha = probe_at_layer(feats, labels, scen, L)
        per_k[k] = {"r2": r2, "alpha": alpha}
        print(f"  k = {k:2d}   R² = {r2:+.4f}   α = {alpha:.2e}")
    # Degradation slope: R² lost per halving of k. Reported in summary.
    r2_at_16 = per_k[16]["r2"]
    r2_at_2 = per_k[2]["r2"]
    results[prop] = {
        "peak_layer_1indexed": L + 1,
        "by_k": per_k,
        "delta_r2_full_starve": r2_at_2 - r2_at_16,   # negative = degradation
        "fraction_retained_at_k2": (r2_at_2 / r2_at_16) if r2_at_16 > 0 else None,
    }

# --- Summary table ---------------------------------------------------------
print(f"\n{'='*60}\n  SUMMARY\n{'='*60}")
print(f"  {'property':12s}  " + "  ".join(f"k={k:>2d}" for k in K_VALUES) +
      f"   ΔR² (k2-k16)   retained@k2")
for prop in ["elasticity", "friction", "drag"]:
    r = results[prop]
    r2_strs = "  ".join(f"{r['by_k'][k]['r2']:+.3f}" for k in K_VALUES)
    frac = r["fraction_retained_at_k2"]
    frac_s = f"{frac*100:5.1f}%" if frac is not None else "  n/a"
    print(f"  {prop:12s}  {r2_strs}   {r['delta_r2_full_starve']:+.4f}        {frac_s}")

# --- Save -----------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / "temporal_integration_length.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved → {out_path}")

In [ ]:
"""
Shuffled-label sanity check
============================
Identical to the real probing pipeline, but labels are randomly permuted.
Every R² should be near zero. If any layer scores well, something is wrong.

Usage: paste into Colab cell, update DATA_DIR, run.
"""

import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit

# ── Update this to wherever your .npz files live ──
DATA_DIR = "./features6.0"  # <-- CHANGE THIS

PROPERTIES = ["elasticity", "friction", "mass", "drag"]
N_LAYERS = 12
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS = np.logspace(-3, 6, 50)
SHUFFLE_SEED = 9999


def probe(X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_te = scaler.transform(X_test)
    model = RidgeCV(alphas=ALPHAS, scoring="r2")
    model.fit(X_tr, y_train)
    return r2_score(y_test, model.predict(X_te))


for prop in PROPERTIES:
    print(f"\n{'='*60}")
    print(f"  SHUFFLED CONTROL — {prop.upper()}")
    print(f"{'='*60}")

    data = np.load(f"{DATA_DIR}/vjepa_{prop}.npz")
    features = data["features"]
    labels = data["labels"]
    scenarios = data["scenario_ids"]

    rng = np.random.default_rng(SHUFFLE_SEED)
    shuffled_labels = labels.copy()
    rng.shuffle(shuffled_labels)

    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(gss.split(features, labels, groups=scenarios))

    for layer in range(N_LAYERS):
        X = features[:, layer, :]
        r2 = probe(
            X[train_idx], X[test_idx],
            shuffled_labels[train_idx], shuffled_labels[test_idx],
        )
        flag = " ⚠️ LEAK" if r2 > 0.05 else ""
        print(f"  Layer {layer+1:2d}  →  R² = {r2:.4f}{flag}")

    # CLIP
    clip = np.load(f"{DATA_DIR}/clip_{prop}.npz")
    clip_scenarios = clip["scenario_ids"]
    shuffled_clip = clip["labels"].copy()
    rng2 = np.random.default_rng(SHUFFLE_SEED)
    rng2.shuffle(shuffled_clip)

    gss_clip = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    clip_tr, clip_te = next(gss_clip.split(clip["features"], clip["labels"], groups=clip_scenarios))

    r2 = probe(
        clip["features"][clip_tr], clip["features"][clip_te],
        shuffled_clip[clip_tr], shuffled_clip[clip_te],
    )
    print(f"\n  CLIP     →  R² = {r2:.4f}{'  ⚠️ LEAK' if r2 > 0.05 else ''}")

    # DINOv2
    dino = np.load(f"{DATA_DIR}/dino_{prop}.npz")
    dino_scenarios = dino["scenario_ids"]
    shuffled_dino = dino["labels"].copy()
    rng3 = np.random.default_rng(SHUFFLE_SEED)
    rng3.shuffle(shuffled_dino)

    gss_dino = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    dino_tr, dino_te = next(gss_dino.split(dino["features"], dino["labels"], groups=dino_scenarios))

    r2 = probe(
        dino["features"][dino_tr], dino["features"][dino_te],
        shuffled_dino[dino_tr], shuffled_dino[dino_te],
    )
    print(f"  DINOv2   →  R² = {r2:.4f}{'  ⚠️ LEAK' if r2 > 0.05 else ''}")

print(f"\n{'='*60}")
print("  If any line shows ⚠️ LEAK, there's a problem.")
print("  All values should be near 0 (±0.05).")
print(f"{'='*60}")

In [ ]:
# === Probe: Single-Frame-Duplicated V-JEPA at Peak Layer ===
import json
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

SAVE_DIR_SF = "./features6.0_singleframe"
OUT_DIR = Path("./probing_results6.0")
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS = np.logspace(-3, 6, 50)

# Update with the peak layers from your main probing run
PEAK_LAYERS = {"elasticity": 8, "friction": 12, "drag": 11}

def probe(X_tr, X_te, y_tr, y_te):
    s = StandardScaler()
    X_tr = s.fit_transform(X_tr)
    X_te = s.transform(X_te)
    m = RidgeCV(alphas=ALPHAS, scoring="r2").fit(X_tr, y_tr)
    return r2_score(y_te, m.predict(X_te))

print("=== Single-Frame-Duplicated Control (Last Frame ×16) ===\n")
results = {}
for prop, peak in PEAK_LAYERS.items():
    data = np.load(f"{SAVE_DIR_SF}/vjepa_lastframe_{prop}.npz")
    feats = data["features"][:, peak - 1, :]
    labels = data["labels"]
    scenarios = data["scenario_ids"]

    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    tr, te = next(gss.split(feats, labels, groups=scenarios))

    r2 = probe(feats[tr], feats[te], labels[tr], labels[te])
    #flag = "⚠️ LEAK" if r2 > 0.10 else "✓"
    flag = ""
    print(f"  {prop:12s} (peak L{peak}, last-frame×16)  →  R² = {r2:.4f}  {flag}")
    results[prop] = {"peak_layer": peak, "r2_lastframe": float(r2)}

OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "singleframe_control.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved → {OUT_DIR / 'singleframe_control.json'}")

In [ ]:
# === Subspace Geometry: Principal Angles Between Property Probes ===
# At each V-JEPA layer, fits three ridge probes (elasticity, friction, drag),
# then measures the angle between their weight vectors. Small angles at deep
# layers = the three properties are read from a shared subspace (the
# "latent physical constant" category predicted by the hierarchy thesis).

import json
import numpy as np
from pathlib import Path
from scipy.linalg import subspace_angles
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

PROPERTIES = ["elasticity", "friction", "drag"]
N_LAYERS = 12
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS_CV = np.logspace(-3, 6, 50)             # for the CV-selected ridge
ALPHAS_FIXED = [1.0, 1e2, 1e4]                  # for the λ-sensitivity check

DATA_DIR = Path("./features6.0")
OUT_DIR = Path("./probing_results6.0")


def load_features(prop):
    """Load V-JEPA per-layer features for one property."""
    data = np.load(DATA_DIR / f"vjepa_{prop}.npz")
    return data["features"], data["labels"], data["scenario_ids"]


def fit_weights(feats, labels, scen, alpha_mode):
    """
    Fit a probe at every layer, return weight matrix W of shape (N_LAYERS, d).

    alpha_mode: "cv" → RidgeCV picks alpha per layer.
                float → Ridge with that fixed alpha.
    Same train/test scenario split as the main probing pipeline (random_state=42),
    so weights live in the same standardized basis the main probes use.
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    tr, _ = next(gss.split(feats, labels, groups=scen))

    d = feats.shape[-1]
    W = np.zeros((N_LAYERS, d))
    for layer in range(N_LAYERS):
        X_tr = feats[tr, layer, :]
        y_tr = labels[tr]
        # Standardize: this is the basis ridge actually optimizes in,
        # so the resulting w reads directions in this scaled space.
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        if alpha_mode == "cv":
            model = RidgeCV(alphas=ALPHAS_CV, scoring="r2").fit(X_tr_s, y_tr)
        else:
            model = Ridge(alpha=alpha_mode).fit(X_tr_s, y_tr)
        # sklearn returns w of shape (d,) for 1-D targets; L2-normalize so
        # only the *direction* matters. (Sign-invariance is handled later
        # by subspace_angles, which compares the lines spanned, not the rays.)
        w = model.coef_
        W[layer] = w / (np.linalg.norm(w) + 1e-12)
    return W


def pairwise_angles(W_E, W_F, W_D):
    """
    For each layer, compute the three pairwise principal angles between the
    1-D subspaces spanned by each property's weight vector, then average.
    subspace_angles returns angles in [0, π/2]; smaller = more aligned.
    """
    mean_ang = np.zeros(N_LAYERS)
    pairs = np.zeros((N_LAYERS, 3))  # columns: E-F, E-D, F-D
    for L in range(N_LAYERS):
        # Each weight is shape (d,); subspace_angles wants column vectors
        # representing 1-D subspaces, i.e. shape (d, 1).
        wE = W_E[L][:, None]
        wF = W_F[L][:, None]
        wD = W_D[L][:, None]
        a_EF = subspace_angles(wE, wF)[0]
        a_ED = subspace_angles(wE, wD)[0]
        a_FD = subspace_angles(wF, wD)[0]
        pairs[L] = [a_EF, a_ED, a_FD]
        mean_ang[L] = pairs[L].mean()
    return mean_ang, pairs


# --- Run: one weight matrix per property per alpha mode --------------------
print(f"{'='*60}\n  Subspace geometry: principal angles per layer\n{'='*60}")
features_by_prop = {p: load_features(p) for p in PROPERTIES}

results = {}
for tag, alpha_mode in [("cv", "cv")] + [(f"a{a:g}", a) for a in ALPHAS_FIXED]:
    print(f"\n--- alpha_mode = {tag} ---")
    Ws = {p: fit_weights(*features_by_prop[p], alpha_mode=alpha_mode)
          for p in PROPERTIES}
    mean_ang, pairs = pairwise_angles(Ws["elasticity"], Ws["friction"], Ws["drag"])
    # Print in degrees for readability; π/2 = 90° = orthogonal.
    for L in range(N_LAYERS):
        deg = np.degrees(mean_ang[L])
        print(f"  Layer {L+1:2d}  mean angle = {deg:5.1f}°  "
              f"(E-F {np.degrees(pairs[L,0]):5.1f}°, "
              f"E-D {np.degrees(pairs[L,1]):5.1f}°, "
              f"F-D {np.degrees(pairs[L,2]):5.1f}°)")
    results[tag] = {
        "mean_angle_rad": mean_ang.tolist(),
        "pairwise_angles_rad": pairs.tolist(),  # columns: E-F, E-D, F-D
        "alpha_mode": str(alpha_mode),
    }

# --- Save -----------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / "subspace_geometry.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved → {out_path}")

In [ ]:
# === Subspace Geometry: Baselines (CLIP, DINOv2) + Random-Vector Null ===
# Extends the V-JEPA subspace-angle analysis with two controls:
#   (1) Per-layer angles for CLIP and DINOv2, to test whether the V-JEPA
#       angles are model-specific or just a generic high-dimensional effect.
#   (2) A random-vector null: angle distribution from triples of random unit
#       vectors in R^d. In high dimensions this concentrates near 90°, so
#       any meaningful deviation by V-JEPA must clear this floor.

import json
import numpy as np
from pathlib import Path
from scipy.linalg import subspace_angles
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

PROPERTIES = ["elasticity", "friction", "drag"]
N_LAYERS = 12
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS_CV = np.logspace(-3, 6, 50)
N_NULL_TRIALS = 2000  # number of random unit-vector triples per dimension

DATA_DIR = Path("./features6.0")
OUT_DIR = Path("./probing_results6.0")


def load_features(prefix, prop):
    """Load per-layer features for one model+property. Same format across models:
    features (n_videos, n_layers, d), labels (n_videos,), scenario_ids (n_videos,)."""
    data = np.load(DATA_DIR / f"{prefix}_{prop}.npz")
    return data["features"], data["labels"], data["scenario_ids"]


def fit_weights(feats, labels, scen):
    """Fit RidgeCV at every layer (matching the main probing pipeline exactly),
    return L2-normalized weight matrix W of shape (N_LAYERS, d)."""
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    tr, _ = next(gss.split(feats, labels, groups=scen))

    d = feats.shape[-1]
    W = np.zeros((N_LAYERS, d))
    for layer in range(N_LAYERS):
        # Standardize on train (same basis ridge optimizes in)
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(feats[tr, layer, :])
        model = RidgeCV(alphas=ALPHAS_CV, scoring="r2").fit(X_tr_s, labels[tr])
        w = model.coef_
        W[layer] = w / (np.linalg.norm(w) + 1e-12)
    return W


def pairwise_angles(W_E, W_F, W_D):
    """For each layer, mean of three pairwise principal angles (E-F, E-D, F-D),
    in radians. subspace_angles handles sign-invariance automatically."""
    mean_ang = np.zeros(N_LAYERS)
    pairs = np.zeros((N_LAYERS, 3))
    for L in range(N_LAYERS):
        wE, wF, wD = W_E[L][:, None], W_F[L][:, None], W_D[L][:, None]
        pairs[L] = [
            subspace_angles(wE, wF)[0],
            subspace_angles(wE, wD)[0],
            subspace_angles(wF, wD)[0],
        ]
        mean_ang[L] = pairs[L].mean()
    return mean_ang, pairs


def random_null_distribution(d, n_trials, seed=0):
    """Sample n_trials triples of random unit vectors in R^d, compute mean of the
    three pairwise angles for each triple. Returns an array of length n_trials.
    In high d this concentrates near 90° — that's the bar real probes must clear."""
    rng = np.random.default_rng(seed)
    means = np.zeros(n_trials)
    for i in range(n_trials):
        # Gaussian → normalize gives uniform direction on the unit sphere
        v = rng.standard_normal((3, d))
        v /= np.linalg.norm(v, axis=1, keepdims=True)
        # |inner product| because we compare lines, not rays (matches subspace_angles)
        c01 = abs(v[0] @ v[1])
        c02 = abs(v[0] @ v[2])
        c12 = abs(v[1] @ v[2])
        # clip for numerical safety before arccos
        means[i] = np.mean(np.arccos(np.clip([c01, c02, c12], 0.0, 1.0)))
    return means


# --- Per-model angle curves ------------------------------------------------
print(f"{'='*60}\n  Subspace angles: V-JEPA vs CLIP vs DINOv2\n{'='*60}")
results = {}

for tag, prefix in [("vjepa", "vjepa"),
                    ("clip", "clip_perlayer"),
                    ("dino", "dino_perlayer")]:
    print(f"\n--- Model: {tag.upper()} ---")
    feats_by_prop = {p: load_features(prefix, p) for p in PROPERTIES}
    Ws = {p: fit_weights(*feats_by_prop[p]) for p in PROPERTIES}
    mean_ang, pairs = pairwise_angles(Ws["elasticity"], Ws["friction"], Ws["drag"])
    for L in range(N_LAYERS):
        print(f"  Layer {L+1:2d}  mean angle = {np.degrees(mean_ang[L]):5.1f}°  "
              f"(E-F {np.degrees(pairs[L,0]):5.1f}°, "
              f"E-D {np.degrees(pairs[L,1]):5.1f}°, "
              f"F-D {np.degrees(pairs[L,2]):5.1f}°)")
    # Cache the embed dim per model — needed for the matched-d null below.
    d = next(iter(feats_by_prop.values()))[0].shape[-1]
    results[tag] = {
        "mean_angle_rad": mean_ang.tolist(),
        "pairwise_angles_rad": pairs.tolist(),
        "embed_dim": int(d),
    }

# --- Random-vector null at each model's embedding dimension ----------------
print(f"\n{'='*60}\n  Random-vector null (n={N_NULL_TRIALS} triples per dim)\n{'='*60}")
unique_dims = sorted({results[m]["embed_dim"] for m in results})
nulls = {}
for d in unique_dims:
    samples = random_null_distribution(d, N_NULL_TRIALS, seed=0)
    deg = np.degrees(samples)
    nulls[str(d)] = {
        "mean_deg": float(deg.mean()),
        "std_deg": float(deg.std()),
        "p2_5_deg": float(np.percentile(deg, 2.5)),
        "p97_5_deg": float(np.percentile(deg, 97.5)),
        "samples_rad": samples.tolist(),
    }
    print(f"  d = {d:4d}   mean = {deg.mean():.2f}°   std = {deg.std():.2f}°   "
          f"95% CI = [{np.percentile(deg, 2.5):.2f}°, {np.percentile(deg, 97.5):.2f}°]")

# --- Summary: how far does each model deviate from the null? ---------------
print(f"\n{'='*60}\n  Deviation from null (most-aligned layer per model)\n{'='*60}")
for m, r in results.items():
    null = nulls[str(r["embed_dim"])]
    angles_deg = np.degrees(np.array(r["mean_angle_rad"]))
    min_layer = int(np.argmin(angles_deg))
    min_deg = float(angles_deg[min_layer])
    # z-score: how many null-stds below the null mean is this layer's angle?
    z = (min_deg - null["mean_deg"]) / null["std_deg"]
    print(f"  {m:6s}  min angle = {min_deg:.2f}° at L{min_layer+1:2d}   "
          f"null mean = {null['mean_deg']:.2f}°   "
          f"deviation = {min_deg - null['mean_deg']:+.2f}°   z = {z:+.1f}")

# --- Save -----------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / "subspace_geometry_baselines.json"
# Don't dump the full sample arrays into the main JSON — too large; just stats.
nulls_summary = {k: {kk: vv for kk, vv in v.items() if kk != "samples_rad"}
                 for k, v in nulls.items()}
with open(out_path, "w") as f:
    json.dump({"models": results, "null": nulls_summary}, f, indent=2)
# Save the raw null samples separately for histogram plotting later.
np.savez(OUT_DIR / "subspace_null_samples.npz",
         **{f"d{d}": np.array(nulls[str(d)]["samples_rad"]) for d in unique_dims})
print(f"\nSaved → {out_path}")
print(f"Saved → {OUT_DIR / 'subspace_null_samples.npz'}")

In [ ]:
# === Experiment 1: Causal ablation of probe direction (CORRECTED) ===
# Two questions:
#   (a) NECESSITY: if we remove the probe's direction from features and re-fit a
#       fresh probe on the orthogonal complement, can we still recover the property?
#       If yes -> property is in a high-dimensional code (Joseph-style finding).
#       If no  -> the rank-1 direction was genuinely necessary.
#   (b) SPECIFICITY: ablate property A's direction from property B's features —
#       does B's probe still work? If yes, A and B's directions are nearly orthogonal.
#       If no, they share representational subspace.

import json
import numpy as np
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit

PROPERTIES = ["elasticity", "friction", "drag"]
PEAK_LAYERS_0IDX = {"elasticity": 7, "friction": 11, "drag": 10}
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS = np.logspace(-3, 6, 50)

DATA_DIR = Path("./features6.0")
OUT_DIR = Path("./probing_results6.0")


def load_vjepa(prop):
    data = np.load(DATA_DIR / f"vjepa_{prop}.npz")
    return data["features"], data["labels"], data["scenario_ids"]


def fit_probe_with_direction(X_tr_raw, y_tr):
    """Fit RidgeCV in standardized basis, return scaler, model, and unit-norm
    weight direction (in the standardized basis where the probe was fit)."""
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr_raw)
    model = RidgeCV(alphas=ALPHAS, scoring="r2").fit(X_tr_s, y_tr)
    w = model.coef_
    w_unit = w / (np.linalg.norm(w) + 1e-12)
    return scaler, model, w_unit


def project_out(X_standardized, w_unit):
    """Orthogonally remove the rank-1 subspace spanned by w_unit from X."""
    return X_standardized - np.outer(X_standardized @ w_unit, w_unit)


# --- Part (a): necessity test via fresh-probe-on-ablated-features --------
print(f"{'='*60}\n  (a) NECESSITY: re-fit probe on orthogonal complement\n{'='*60}")

probes = {}
necessity_results = {}
for prop in PROPERTIES:
    feats, labels, scen = load_vjepa(prop)
    L = PEAK_LAYERS_0IDX[prop]
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    tr, te = next(gss.split(feats, labels, groups=scen))

    X_tr_raw = feats[tr, L, :]
    X_te_raw = feats[te, L, :]
    y_tr, y_te = labels[tr], labels[te]

    # Original probe (the headline result)
    scaler, model, w_unit = fit_probe_with_direction(X_tr_raw, y_tr)
    X_tr_s = scaler.transform(X_tr_raw)
    X_te_s = scaler.transform(X_te_raw)
    r2_baseline = r2_score(y_te, model.predict(X_te_s))

    # Ablate w_unit from BOTH train and test, then fit a FRESH probe.
    # Note: technically we should re-standardize after projection (the projected
    # features have slightly different per-feature variances), but doing so
    # changes the basis again and the ablation is no longer in that basis.
    # The standard practice in the probing literature is to leave it as-is —
    # the projection moves features off the unit-variance manifold by a small
    # amount that doesn't affect the necessity question.
    X_tr_ablated = project_out(X_tr_s, w_unit)
    X_te_ablated = project_out(X_te_s, w_unit)

    fresh_probe = RidgeCV(alphas=ALPHAS, scoring="r2").fit(X_tr_ablated, y_tr)
    r2_after = r2_score(y_te, fresh_probe.predict(X_te_ablated))

    retention_pct = 100.0 * r2_after / r2_baseline if r2_baseline > 0 else float("nan")

    print(f"  {prop:12s} L{L+1:2d}  baseline R² = {r2_baseline:+.4f}  |  "
          f"after ablation + fresh probe = {r2_after:+.4f}  "
          f"(retention = {retention_pct:5.1f}%)")

    probes[prop] = {
        "layer_0idx": L,
        "scaler": scaler,
        "model": model,
        "w_unit": w_unit,
        "test_idx": te,
    }
    necessity_results[prop] = {
        "peak_layer_1idx": L + 1,
        "r2_baseline": float(r2_baseline),
        "r2_after_rank1_ablation": float(r2_after),
        "retention_pct": float(retention_pct) if not np.isnan(retention_pct) else None,
    }


# --- Part (b): specificity test via cross-property ablation --------------
# At each property's peak layer, ablate every OTHER property's direction (computed
# from features fit at the SAME layer) and check whether the target property's
# probe still works.
#
# Subtlety: each property's w_unit lives in its own standardized basis (since each
# property has its own scaler). To ablate property A's direction from property B's
# features cleanly, both directions must live in the same basis. We re-fit A's
# probe on B's training features at the relevant layer, with A's labels — but A
# and B are different datasets. So we instead use scaler_B to standardize, and
# project along A's w_unit. This is a small approximation: the bases differ
# slightly because the per-feature means/stds differ between datasets, but the
# encoder is the same and the input distributions are similar (synthetic Pymunk
# physics across all properties), so the bases are nearly aligned. The baseline
# R² printed alongside acts as a sanity check.

print(f"\n{'='*60}\n  (b) SPECIFICITY: cross-property ablation per layer\n{'='*60}")

cross_results = {}
for L_name, L in PEAK_LAYERS_0IDX.items():
    cross_results[f"L{L+1}"] = {}
    print(f"\n--- At L{L+1} (peak layer for {L_name}) ---")

    # Fit probe + extract direction for every property at this layer
    layer_state = {}
    for prop in PROPERTIES:
        feats, labels, scen = load_vjepa(prop)
        gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        tr, te = next(gss.split(feats, labels, groups=scen))
        scaler_p, model_p, w_unit_p = fit_probe_with_direction(feats[tr, L, :], labels[tr])
        layer_state[prop] = {
            "scaler": scaler_p,
            "model": model_p,
            "w_unit": w_unit_p,
            "X_te_raw": feats[te, L, :],
            "y_te": labels[te],
        }

    for target in PROPERTIES:
        scaler_t = layer_state[target]["scaler"]
        model_t = layer_state[target]["model"]
        X_te_s = scaler_t.transform(layer_state[target]["X_te_raw"])
        y_te = layer_state[target]["y_te"]
        r2_baseline = r2_score(y_te, model_t.predict(X_te_s))

        for ablate in PROPERTIES:
            if ablate == target:
                continue
            w_ablate = layer_state[ablate]["w_unit"]
            X_te_s_ablated = project_out(X_te_s, w_ablate)
            r2_ablated = r2_score(y_te, model_t.predict(X_te_s_ablated))
            delta = r2_ablated - r2_baseline
            print(f"  target={target:12s} ablate={ablate:12s}  "
                  f"baseline={r2_baseline:+.4f}  ablated={r2_ablated:+.4f}  "
                  f"Δ={delta:+.4f}")
            cross_results[f"L{L+1}"][f"{target}_ablate_{ablate}"] = {
                "baseline_r2": float(r2_baseline),
                "ablated_r2": float(r2_ablated),
                "delta": float(delta),
            }


# --- Save -----------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "ablation_results.json", "w") as f:
    json.dump({
        "necessity_fresh_probe": necessity_results,
        "specificity_cross_ablation": cross_results,
    }, f, indent=2)
print(f"\nSaved → {OUT_DIR / 'ablation_results.json'}")

In [ ]:
# === Experiment 2: Cross-property probe transfer + within-property extrapolation ===
# (a) Train probe on property A, evaluate on property B's features+labels at same layer.
#     Tests whether properties share representational subspaces.
# (b) Train probe on lower half of label range, test on upper half. Tests whether the
#     representation is continuous vs categorical.

import json
import numpy as np
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit

PROPERTIES = ["elasticity", "friction", "drag"]
PEAK_LAYERS_0IDX = {"elasticity": 7, "friction": 11, "drag": 10}
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS = np.logspace(-3, 6, 50)

DATA_DIR = Path("./features6.0")
OUT_DIR = Path("./probing_results6.0")


def load_vjepa(prop):
    data = np.load(DATA_DIR / f"vjepa_{prop}.npz")
    return data["features"], data["labels"], data["scenario_ids"]


# --- Part (a): cross-property transfer ----------------------------------
# At each layer L ∈ {peak layers}, train probe on source, test on target's videos
# (using target's labels). Source and target are different datasets, so the probe
# never sees the target distribution during fitting.

print(f"{'='*60}\n  (a) Cross-property probe transfer\n{'='*60}")

transfer_results = {}
for L_name, L in PEAK_LAYERS_0IDX.items():
    print(f"\n--- Layer L{L+1} (peak for {L_name}) ---")
    transfer_results[f"L{L+1}"] = {}

    # Fit a probe per property at this layer
    layer_probes = {}
    for prop in PROPERTIES:
        feats, labels, scen = load_vjepa(prop)
        gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        tr, te = next(gss.split(feats, labels, groups=scen))

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(feats[tr, L, :])
        model = RidgeCV(alphas=ALPHAS, scoring="r2").fit(X_tr_s, labels[tr])
        layer_probes[prop] = {
            "scaler": scaler, "model": model,
            "feats_te": feats[te, L, :], "labels_te": labels[te],
            "test_idx": te,
        }
        # Within-property baseline
        r2_self = r2_score(labels[te], model.predict(scaler.transform(feats[te, L, :])))
        print(f"  {prop:12s} -> {prop:12s} (within-property)  R² = {r2_self:+.4f}")
        transfer_results[f"L{L+1}"][f"{prop}_to_{prop}"] = float(r2_self)

    # Cross-property: probe trained on source, evaluated on target's test features+labels
    for source in PROPERTIES:
        for target in PROPERTIES:
            if source == target:
                continue
            scaler_s = layer_probes[source]["scaler"]
            model_s = layer_probes[source]["model"]
            X_target = layer_probes[target]["feats_te"]
            y_target = layer_probes[target]["labels_te"]

            # Apply source's scaler to target's features (mismatch is informative —
            # if target's feature distribution differs, that's part of what we measure)
            X_target_s = scaler_s.transform(X_target)
            y_pred = model_s.predict(X_target_s)
            r2 = r2_score(y_target, y_pred)

            # Pearson correlation as a label-scale-invariant secondary metric.
            # Cross-property R² is hard to interpret when label units differ
            # (elasticity ∈ [0,1], drag is a coefficient, friction is a coefficient
            # — different scales mean R² can be negative even if the probe captures
            # the *ranking*). Pearson r tells you whether the predicted values are
            # MONOTONIC with the target labels.
            from scipy.stats import pearsonr
            r_pear, _ = pearsonr(y_pred, y_target)

            print(f"  {source:12s} -> {target:12s}  R² = {r2:+.4f}  "
                  f"Pearson r = {r_pear:+.4f}")
            transfer_results[f"L{L+1}"][f"{source}_to_{target}"] = {
                "r2": float(r2),
                "pearson_r": float(r_pear),
            }


# --- Part (b): held-out-value extrapolation (CORRECTED) -----------------
# Hold out the lowest and highest property values entirely. Train on middle values.
# This breaks scenario-cleanness (test scenarios appear in train at other property
# levels) but tests the right question: does the probe extrapolate to unseen
# property magnitudes?
#
# R² is undefined when y_test has zero variance (all the same label). Use MAE,
# normalized by the std of the training labels for cross-property comparability.

import json
import numpy as np
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit

PROPERTIES = ["elasticity", "friction", "drag"]
PEAK_LAYERS_0IDX = {"elasticity": 7, "friction": 11, "drag": 10}
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS = np.logspace(-3, 6, 50)

DATA_DIR = Path("./features6.0")
OUT_DIR = Path("./probing_results6.0")


def load_vjepa(prop):
    data = np.load(DATA_DIR / f"vjepa_{prop}.npz")
    return data["features"], data["labels"], data["scenario_ids"]


print(f"{'='*60}\n  (b) Held-out-value extrapolation (corrected)\n{'='*60}")

extrap_results = {}
for prop in PROPERTIES:
    feats, labels, scen = load_vjepa(prop)
    L = PEAK_LAYERS_0IDX[prop]

    unique_vals = np.sort(np.unique(labels))
    if len(unique_vals) < 3:
        print(f"  [{prop}] only {len(unique_vals)} unique values, skipping")
        continue

    middle_vals = unique_vals[1:-1]
    lowest_val = float(unique_vals[0])
    highest_val = float(unique_vals[-1])

    train_mask = np.isin(labels, middle_vals)
    test_low_mask = labels == unique_vals[0]
    test_high_mask = labels == unique_vals[-1]

    X_tr = feats[train_mask, L, :]
    y_tr = labels[train_mask]
    X_te_low = feats[test_low_mask, L, :]
    y_te_low = labels[test_low_mask]
    X_te_high = feats[test_high_mask, L, :]
    y_te_high = labels[test_high_mask]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    model = RidgeCV(alphas=ALPHAS, scoring="r2").fit(X_tr_s, y_tr)

    pred_low = model.predict(scaler.transform(X_te_low))
    pred_high = model.predict(scaler.transform(X_te_high))

    # MAE in raw units
    mae_low = float(np.mean(np.abs(pred_low - y_te_low)))
    mae_high = float(np.mean(np.abs(pred_high - y_te_high)))

    # MAE normalized by training-label std — comparable across properties.
    # A normalized MAE < 1 means the probe extrapolates better than predicting
    # the train mean; > 1 means it does worse than the no-information baseline.
    train_std = float(np.std(y_tr))
    mae_low_norm = mae_low / train_std
    mae_high_norm = mae_high / train_std

    # Also report the mean predicted value at each held-out level. If the probe
    # is extrapolating, mean(pred_low) should be < mean(pred_high). If the probe
    # is just predicting the train mean (no extrapolation), they'll be similar.
    pred_low_mean = float(np.mean(pred_low))
    pred_high_mean = float(np.mean(pred_high))
    train_mean = float(np.mean(y_tr))

    # Standard interpolating split for reference
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    tr_i, te_i = next(gss.split(feats, labels, groups=scen))
    scaler_i = StandardScaler()
    X_tr_i_s = scaler_i.fit_transform(feats[tr_i, L, :])
    model_i = RidgeCV(alphas=ALPHAS, scoring="r2").fit(X_tr_i_s, labels[tr_i])
    r2_interp = float(r2_score(labels[te_i], model_i.predict(scaler_i.transform(feats[te_i, L, :]))))

    print(f"\n  {prop:12s} L{L+1:2d}  (interp R² for ref = {r2_interp:+.4f})")
    print(f"    train: {len(middle_vals)} values × ~{int(train_mask.sum() / len(middle_vals))} videos = {int(train_mask.sum())} total, "
          f"label range [{middle_vals[0]:.3f}, {middle_vals[-1]:.3f}], mean={train_mean:.3f}")
    print(f"    held-out LOW  (label = {lowest_val:.3f}, n={int(test_low_mask.sum())}):  "
          f"MAE = {mae_low:.4f}  ({mae_low_norm:.2f} train-σ)  pred mean = {pred_low_mean:.3f}")
    print(f"    held-out HIGH (label = {highest_val:.3f}, n={int(test_high_mask.sum())}): "
          f"MAE = {mae_high:.4f}  ({mae_high_norm:.2f} train-σ)  pred mean = {pred_high_mean:.3f}")

    extrap_results[prop] = {
        "peak_layer_1idx": L + 1,
        "train_values": [float(v) for v in middle_vals],
        "train_mean_label": train_mean,
        "train_label_std": train_std,
        "lowest_held_out": lowest_val,
        "highest_held_out": highest_val,
        "n_train": int(train_mask.sum()),
        "n_test_low": int(test_low_mask.sum()),
        "n_test_high": int(test_high_mask.sum()),
        "r2_interpolation_reference": r2_interp,
        "mae_extrap_low": mae_low,
        "mae_extrap_high": mae_high,
        "mae_extrap_low_normalized": mae_low_norm,
        "mae_extrap_high_normalized": mae_high_norm,
        "pred_mean_low": pred_low_mean,
        "pred_mean_high": pred_high_mean,
    }


OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / "extrapolation_results.json"
with open(out_path, "w") as f:
    json.dump(extrap_results, f, indent=2)
print(f"\nSaved → {out_path}")

In [ ]:
# === Cell A v2: Robust patch-layout verification ===
# Zero out a SPECIFIC SPATIAL REGION (top-left quadrant) of every frame, then
# check which tokens are most affected. Different layouts predict different
# patterns:
#
#   TEMPORAL-MAJOR (T, H, W): for each of 8 temporal chunks of 576 tokens,
#     the first 12 rows of 24 columns (top half) should be affected.
#     I.e., within each 576-token chunk, tokens at h<12 are affected.
#
#   SPATIAL-MAJOR (H, W, T): the first 12*24*8 = 2304 tokens correspond to
#     h<12, and ALL of them are affected. Tokens 2304..4607 (h>=12) are
#     untouched.
#
# These predictions are very different — half the sequence affected (spatial-
# major) vs. periodic stripes throughout (temporal-major). The signal is much
# bigger than 13%.

import torch
import numpy as np

NUM_TEMPORAL = 8
NUM_SPATIAL_H = 24
NUM_SPATIAL_W = 24
NUM_PATCHES_EXPECTED = NUM_TEMPORAL * NUM_SPATIAL_H * NUM_SPATIAL_W  # 4608
PATCH_PIXEL = 16  # ViT-B/16 spatial patch size at 384x384 input

with torch.no_grad():
    test_frames = load_single_video_from_npz(list(PROPERTIES.values())[0], 0)
    # Note: video is (64, 224, 224, 3); preprocess resizes to 384x384.
    # We zero a region in the 224x224 source that will land in the top-half of
    # the 384x384 input after bilinear resize. Top half of 224x224 = top half
    # of 384x384 after resize, which corresponds to spatial patches with h<12.

# Reference (no zeroing)
def first_block_tokens(model, frames):
    captured = {}
    def hook_fn(m, i, o):
        captured["x"] = (o[0] if isinstance(o, tuple) else o).detach().cpu()
    h = model.blocks[0].register_forward_hook(hook_fn)
    try:
        with torch.no_grad():
            video_tensor = preprocess_video(frames)
            model(video_tensor)
            del video_tensor
            torch.cuda.empty_cache()
    finally:
        h.remove()
    return captured["x"].squeeze(0)  # (4608, embed_dim)

ref = first_block_tokens(model, test_frames)

# Zero the TOP HALF of every frame (spatial intervention)
top_zeroed = test_frames.copy()
top_zeroed[:, :112, :, :] = 0  # top 112 of 224 rows -> top half after resize
toks_top = first_block_tokens(model, top_zeroed)

# Per-token L2 difference from reference
diff = (toks_top - ref).norm(dim=-1).numpy()  # (4608,)

assert diff.shape[0] == NUM_PATCHES_EXPECTED

# --- Test the SPATIAL-MAJOR hypothesis: tokens 0..2303 are h<12, tokens 2304..4607 are h>=12 ---
# Under this hypothesis, the first 2304 tokens (h<12) should have HUGE diff;
# tokens 2304..4607 (h>=12) should have tiny diff.
sm_first_half = diff[:NUM_PATCHES_EXPECTED // 2].mean()
sm_second_half = diff[NUM_PATCHES_EXPECTED // 2:].mean()

# --- Test the TEMPORAL-MAJOR hypothesis: tokens are 8 chunks of 576, ---
# within each 576-chunk, the first 12*24=288 are h<12.
# So affected tokens are: for each t in 0..7, indices [t*576 : t*576+288].
# Untouched: [t*576+288 : (t+1)*576].
tm_affected_idx = np.concatenate([np.arange(t*576, t*576 + 288) for t in range(NUM_TEMPORAL)])
tm_untouched_idx = np.concatenate([np.arange(t*576 + 288, (t+1)*576) for t in range(NUM_TEMPORAL)])
tm_affected = diff[tm_affected_idx].mean()
tm_untouched = diff[tm_untouched_idx].mean()

print("Top half of every frame zeroed. Per-token L2 difference from reference:")
print()
print("Hypothesis SPATIAL-MAJOR — tokens flattened as (H, W, T):")
print(f"  first half of sequence  (predicts h<12, AFFECTED)  = {sm_first_half:.3f}")
print(f"  second half of sequence (predicts h>=12, UNTOUCHED) = {sm_second_half:.3f}")
print(f"  ratio: {sm_first_half / max(sm_second_half, 1e-9):.1f}x  (predicts >>1)")
print()
print("Hypothesis TEMPORAL-MAJOR — tokens flattened as (T, H, W):")
print(f"  affected indices  (h<12 within each temporal chunk) = {tm_affected:.3f}")
print(f"  untouched indices (h>=12 within each temporal chunk) = {tm_untouched:.3f}")
print(f"  ratio: {tm_affected / max(tm_untouched, 1e-9):.1f}x  (predicts >>1)")
print()

# Whichever hypothesis gives a much bigger affected/untouched ratio is correct.
sm_ratio = sm_first_half / max(sm_second_half, 1e-9)
tm_ratio = tm_affected / max(tm_untouched, 1e-9)

if sm_ratio > 3.0 and tm_ratio < 1.5:
    PATCH_LAYOUT = "spatial_major"
    print(f"✓ Layout: SPATIAL-MAJOR (sm_ratio={sm_ratio:.1f}, tm_ratio={tm_ratio:.1f})")
elif tm_ratio > 3.0 and sm_ratio < 1.5:
    PATCH_LAYOUT = "temporal_major"
    print(f"✓ Layout: TEMPORAL-MAJOR (sm_ratio={sm_ratio:.1f}, tm_ratio={tm_ratio:.1f})")
else:
    PATCH_LAYOUT = "unknown"
    print(f"✗ AMBIGUOUS: sm_ratio={sm_ratio:.1f}, tm_ratio={tm_ratio:.1f}")
    print("  Neither hypothesis cleanly explains the activation pattern.")
    print("  Possible causes: V-JEPA has prefix/register tokens, or interleaved")
    print("  layout, or the zeroing didn't propagate through preprocess properly.")
    print("  Investigate before running Cell B.")

del test_frames, top_zeroed, ref, toks_top
torch.cuda.empty_cache()

In [ ]:
print(f"PATCH_LAYOUT = {PATCH_LAYOUT}")
# Should print: PATCH_LAYOUT = temporal_major

In [ ]:
# === Cell B: Extract per-patch V-JEPA features at peak layer per property ===
# For #5 (spatial localization) and #6 (temporal localization).
# Saves shape (N, 8, 24, 24, 768) per property, fp16, np.savez_compressed.
# Only the peak layer is extracted per property to keep storage manageable.
#
# Storage estimate: 5.3 GB per property at fp16 (before compression).
# Compressed should land at 3-4 GB per property given feature redundancy.

import io, os, zipfile
import numpy as np
import torch
from tqdm import tqdm

assert PATCH_LAYOUT in ("temporal_major", "spatial_major"), \
    "Run Cell A first to determine V-JEPA patch layout."

# Peak layers from headline pipeline (0-indexed)
PEAK_LAYERS_0IDX = {
    "elasticity": 7,    # L8 in 1-indexed
    "friction":   11,   # L12
    "drag":       10,   # L11
}

NUM_TEMPORAL = 8
NUM_SPATIAL_H = 24
NUM_SPATIAL_W = 24
NUM_PATCHES_EXPECTED = NUM_TEMPORAL * NUM_SPATIAL_H * NUM_SPATIAL_W  # 4608

SAVE_DIR_PP = "./features6.0_perpatch"
os.makedirs(SAVE_DIR_PP, exist_ok=True)


def extract_perpatch_at_layer(model, video_tensor, layer_idx_0):
    """
    Return per-patch features at layer_idx_0 for one video.
    Output shape: (8, 24, 24, embed_dim) regardless of native token layout.
    """
    captured = {}
    def hook_fn(m, i, o):
        captured["x"] = (o[0] if isinstance(o, tuple) else o).detach().cpu()
    h = model.blocks[layer_idx_0].register_forward_hook(hook_fn)
    try:
        with torch.no_grad():
            model(video_tensor)
    finally:
        h.remove()

    feat = captured["x"].squeeze(0)  # (num_patches, embed_dim)
    assert feat.shape[0] == NUM_PATCHES_EXPECTED, \
        f"Patch count mismatch at layer {layer_idx_0}: got {feat.shape[0]}"

    if PATCH_LAYOUT == "temporal_major":
        # (T, H, W, d)
        feat = feat.reshape(NUM_TEMPORAL, NUM_SPATIAL_H, NUM_SPATIAL_W, -1)
    else:  # spatial_major
        # (H, W, T, d) -> (T, H, W, d)
        feat = feat.reshape(NUM_SPATIAL_H, NUM_SPATIAL_W, NUM_TEMPORAL, -1).permute(2, 0, 1, 3)

    return feat.numpy()  # (8, 24, 24, embed_dim) fp32


for prop, path in PROPERTIES.items():
    if prop not in PEAK_LAYERS_0IDX:
        print(f"[skip] {prop} has no peak layer assigned (mass goes to appendix)")
        continue

    L = PEAK_LAYERS_0IDX[prop]
    save_path = os.path.join(SAVE_DIR_PP, f"vjepa_perpatch_{prop}_L{L+1}.npz")
    if os.path.exists(save_path):
        print(f"[skip] {save_path} exists")
        continue

    print(f"\n=== {prop.upper()} — extracting per-patch features at L{L+1} (idx {L}) ===")

    labels_key, labels = load_labels_from_npz(path)
    with zipfile.ZipFile(path, "r") as zf:
        with zf.open("scenario_id.npy") as f:
            scenario_ids = np.load(io.BytesIO(f.read()))

    N = labels.shape[0]

    # Allocate output as fp16 to halve RAM during the loop (~2.5 GB allocation)
    all_features = np.empty(
        (N, NUM_TEMPORAL, NUM_SPATIAL_H, NUM_SPATIAL_W, 768),
        dtype=np.float16,
    )

    for vid_idx, frames in enumerate(tqdm(
        stream_videos_from_npz(path, N), total=N, desc=f"{prop} L{L+1}"
    )):
        video_tensor = preprocess_video(frames)
        feat_fp32 = extract_perpatch_at_layer(model, video_tensor, L)  # (8, 24, 24, 768)
        all_features[vid_idx] = feat_fp32.astype(np.float16)
        del video_tensor, feat_fp32
        torch.cuda.empty_cache()

    np.savez_compressed(
        save_path,
        features=all_features,
        labels=labels,
        scenario_ids=scenario_ids,
        property_name=prop,
        peak_layer_1idx=np.int32(L + 1),
        patch_layout=PATCH_LAYOUT,
        feature_dtype="float16",
    )
    size_gb = os.path.getsize(save_path) / 1e9
    print(f"  saved {save_path} ({size_gb:.2f} GB)")

    del all_features
    import gc; gc.collect()

print("\nPer-patch extraction complete.")

In [ ]:
PEAK_LAYERS_0IDX = {
    "elasticity": 7,    # L8 in 1-indexed
    "friction":   11,   # L12
    "drag":       10,   # L11
}

In [ ]:
# === Cell C v2: Memory-safe sanity check ===
# Stream per-video pooled features instead of loading the entire array at once.
# Compares pooled per-patch features against the headline pipeline's L8 features
# without ever holding more than one fp32 video in RAM beyond the file mmaps.

import numpy as np

prop = "elasticity"
L = PEAK_LAYERS_0IDX[prop]

pp_path = f"./features6.0_perpatch/vjepa_perpatch_{prop}_L{L+1}.npz"
hd_path = f"./features6.0/vjepa_{prop}.npz"

# np.load on .npz returns lazy NpzFile objects; the underlying .npy entries
# get loaded only when you index them. We still touch the full feature array
# below, but in chunks, so peak RAM stays low.
pp = np.load(pp_path)
hd = np.load(hd_path)

pp_feat = pp["features"]   # shape (N, 8, 24, 24, 768) fp16 — lazy
hd_feat = hd["features"]   # shape (N, 12, 768) fp32 — lazy

print(f"Per-patch shape: {pp_feat.shape}, dtype: {pp_feat.dtype}")
print(f"Headline shape:  {hd_feat.shape}, dtype: {hd_feat.dtype}")

N = pp_feat.shape[0]
assert hd_feat.shape[0] == N, "Per-patch and headline files have different N"

# Process in chunks of 50 videos. At fp32 expansion this is ~50 * 8*24*24*768*4
# = 0.18 GB per chunk — comfortably small.
CHUNK = 50
diffs = np.empty(N, dtype=np.float32)
norms = np.empty(N, dtype=np.float32)

for start in range(0, N, CHUNK):
    end = min(start + CHUNK, N)
    pp_chunk = pp_feat[start:end].astype(np.float32)              # (chunk, 8, 24, 24, 768)
    pp_pooled = pp_chunk.mean(axis=(1, 2, 3))                     # (chunk, 768)
    hd_chunk = hd_feat[start:end, L, :].astype(np.float32)        # (chunk, 768)

    diffs[start:end] = np.linalg.norm(pp_pooled - hd_chunk, axis=1)
    norms[start:end] = np.linalg.norm(hd_chunk, axis=1)

    del pp_chunk, pp_pooled, hd_chunk

rel = diffs / (norms + 1e-9)

print(f"\nRelative L2 difference (per video):")
print(f"  median = {np.median(rel):.5f}")
print(f"  max    = {rel.max():.5f}")
print(f"  mean   = {rel.mean():.5f}")

if np.median(rel) < 0.01:
    print("\n✓ Pooled per-patch features match headline within fp16 precision.")
elif np.median(rel) < 0.05:
    print("\n⚠ Small discrepancy. Probably fp16 quantization. Acceptable.")
else:
    print("\n✗ Large mismatch. Something is wrong with patch extraction.")

pp.close()
hd.close()

In [ ]:
# === Cell D v3: Per-spatial-patch probing, function-scoped cleanup ===

import json
import gc
import numpy as np
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit

PROPERTIES_LIST = ["elasticity", "friction", "drag"]
PEAK_LAYERS_1IDX = {"elasticity": 8, "friction": 12, "drag": 11}
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS = np.logspace(-3, 6, 50)
CHUNK = 25

PP_DIR = Path("./features6.0_perpatch")
OUT_DIR = Path("./probing_results6.0")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_TEMPORAL = 8
NUM_SPATIAL_H = 24
NUM_SPATIAL_W = 24


def process_one_property(prop):
    """Wrap per-property work in a function so locals get released at return time."""
    L_1idx = PEAK_LAYERS_1IDX[prop]
    pp_path = PP_DIR / f"vjepa_perpatch_{prop}_L{L_1idx}.npz"
    print(f"\n=== {prop.upper()} L{L_1idx} — spatial probing (576 probes) ===")

    pp = np.load(pp_path)
    try:
        labels = np.array(pp["labels"])
        scenarios = np.array(pp["scenario_ids"])
        N = labels.shape[0]
        feats_full = pp["features"]

        feats_temporal_pooled = np.empty((N, NUM_SPATIAL_H, NUM_SPATIAL_W, 768),
                                          dtype=np.float32)

        print(f"  pooling temporal axis in chunks of {CHUNK}...")
        for start in range(0, N, CHUNK):
            end = min(start + CHUNK, N)
            chunk = feats_full[start:end].astype(np.float32)
            feats_temporal_pooled[start:end] = chunk.mean(axis=1)
            del chunk

        del feats_full
    finally:
        pp.close()
        del pp
    gc.collect()

    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    tr, te = next(gss.split(np.zeros(N), labels, groups=scenarios))
    y_tr, y_te = labels[tr], labels[te]

    r2_grid = np.full((NUM_SPATIAL_H, NUM_SPATIAL_W), np.nan, dtype=np.float32)
    alpha_grid = np.full((NUM_SPATIAL_H, NUM_SPATIAL_W), np.nan, dtype=np.float32)

    for h in range(NUM_SPATIAL_H):
        for w in range(NUM_SPATIAL_W):
            X = feats_temporal_pooled[:, h, w, :]
            scaler = StandardScaler()
            X_tr_s = scaler.fit_transform(X[tr])
            X_te_s = scaler.transform(X[te])
            model = RidgeCV(alphas=ALPHAS, scoring="r2").fit(X_tr_s, y_tr)
            r2_grid[h, w] = r2_score(y_te, model.predict(X_te_s))
            alpha_grid[h, w] = model.alpha_
        print(f"  row {h+1:2d}/{NUM_SPATIAL_H}  "
              f"mean R² = {np.nanmean(r2_grid[h]):+.3f}  "
              f"max = {np.nanmax(r2_grid[h]):+.3f}")

    h_max, w_max = np.unravel_index(np.nanargmax(r2_grid), r2_grid.shape)
    print(f"\n  {prop} L{L_1idx} summary:")
    print(f"    overall mean R² = {np.nanmean(r2_grid):+.4f}")
    print(f"    max patch R²    = {r2_grid[h_max, w_max]:+.4f} at (h={h_max}, w={w_max})")
    print(f"    min patch R²    = {np.nanmin(r2_grid):+.4f}")
    print(f"    max/min ratio   = {r2_grid[h_max, w_max] / max(np.nanmin(r2_grid), 1e-9):.2f}x")

    return {
        "peak_layer_1idx": L_1idx,
        "r2_grid": r2_grid.tolist(),
        "alpha_grid": alpha_grid.tolist(),
        "shape": [NUM_SPATIAL_H, NUM_SPATIAL_W],
    }


# Resume-aware: load any previously-saved results so a crash mid-property
# doesn't force re-running already-completed properties.
results_path = OUT_DIR / "spatial_localization.json"
if results_path.exists():
    with open(results_path) as f:
        spatial_results = json.load(f)
    print(f"Loaded existing results for: {list(spatial_results.keys())}")
else:
    spatial_results = {}

for prop in PROPERTIES_LIST:
    if prop in spatial_results:
        print(f"[skip] {prop} already done")
        continue
    spatial_results[prop] = process_one_property(prop)
    # Save incrementally so partial progress survives a crash
    with open(results_path, "w") as f:
        json.dump(spatial_results, f, indent=2)
    gc.collect()

print(f"\nSaved → {results_path}")

In [ ]:
# === Cell E v3: Per-temporal-patch probing, properly cleaned up between properties ===

import json
import gc
import numpy as np
from pathlib import Path
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit

PROPERTIES_LIST = ["elasticity", "friction", "drag"]
PEAK_LAYERS_1IDX = {"elasticity": 8, "friction": 12, "drag": 11}
TEST_SIZE = 0.2
RANDOM_STATE = 42
ALPHAS = np.logspace(-3, 6, 50)
CHUNK = 25  # smaller chunk = lower transient peak

PP_DIR = Path("./features6.0_perpatch")
OUT_DIR = Path("./probing_results6.0")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_TEMPORAL = 8

temporal_results = {}


def process_one_property(prop):
    """Wrapped in a function so all locals go out of scope at return time —
    the cleanest way to guarantee no lingering references between properties."""
    L_1idx = PEAK_LAYERS_1IDX[prop]
    pp_path = PP_DIR / f"vjepa_perpatch_{prop}_L{L_1idx}.npz"
    print(f"\n=== {prop.upper()} L{L_1idx} — temporal probing (8 probes) ===")

    pp = np.load(pp_path)
    try:
        labels = np.array(pp["labels"])           # force materialize, drop lazy ref
        scenarios = np.array(pp["scenario_ids"])
        N = labels.shape[0]
        feats_full = pp["features"]               # lazy fp16 ref (still inside pp)

        # Final pooled shape: (N, 8, 768) fp32 ≈ 18 MB. Trivial.
        feats_spatial_pooled = np.empty((N, NUM_TEMPORAL, 768), dtype=np.float32)

        print(f"  pooling spatial axes in chunks of {CHUNK}...")
        for start in range(0, N, CHUNK):
            end = min(start + CHUNK, N)
            chunk = feats_full[start:end].astype(np.float32)
            feats_spatial_pooled[start:end] = chunk.mean(axis=(2, 3))
            del chunk

        del feats_full   # drop lazy reference BEFORE closing pp
    finally:
        pp.close()
        del pp           # drop the NpzFile binding too
    gc.collect()

    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    tr, te = next(gss.split(np.zeros(N), labels, groups=scenarios))
    y_tr, y_te = labels[tr], labels[te]

    r2s = np.full(NUM_TEMPORAL, np.nan, dtype=np.float32)
    alphas_used = np.full(NUM_TEMPORAL, np.nan, dtype=np.float32)

    for t in range(NUM_TEMPORAL):
        X = feats_spatial_pooled[:, t, :]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X[tr])
        X_te_s = scaler.transform(X[te])
        model = RidgeCV(alphas=ALPHAS, scoring="r2").fit(X_tr_s, y_tr)
        r2s[t] = r2_score(y_te, model.predict(X_te_s))
        alphas_used[t] = model.alpha_
        print(f"  t={t}  R² = {r2s[t]:+.4f}  α = {alphas_used[t]:.2e}")

    print(f"\n  {prop} L{L_1idx} temporal: peak at t={int(np.argmax(r2s))}, "
          f"max R² = {r2s.max():+.4f}, min R² = {r2s.min():+.4f}")

    return {
        "peak_layer_1idx": L_1idx,
        "r2_per_temporal_patch": r2s.tolist(),
        "alpha_per_temporal_patch": alphas_used.tolist(),
    }


for prop in PROPERTIES_LIST:
    temporal_results[prop] = process_one_property(prop)
    gc.collect()  # cleanup after function returns; all its locals are gone

with open(OUT_DIR / "temporal_localization.json", "w") as f:
    json.dump(temporal_results, f, indent=2)
print(f"\nSaved → {OUT_DIR / 'temporal_localization.json'}")

In [ ]:
import os, zipfile, glob

# Find the zip file under MyDrive/data
candidates = glob.glob("/content/drive/MyDrive/data/*.zip") + glob.glob("/content/drive/MyDrive/data/**/*.zip", recursive=True)
print("Zip files found:")
for c in candidates:
    size_mb = os.path.getsize(c) / 1e6
    print(f"  {c}  ({size_mb:.1f} MB)")

In [ ]:
# === Cell F v3: Spacetime localization figure (self-contained) ===

import json
import io
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path("./probing_results6.0")
PROPERTIES_LIST = ["elasticity", "friction", "drag"]
PEAK_LAYERS_1IDX = {"elasticity": 8, "friction": 12, "drag": 11}

# Self-contained: define paths to the source .npz files locally so this cell
# doesn't depend on cell 46's PROPERTIES dict being in scope.
PROPERTY_NPZ_PATHS = {
    "elasticity": "/content/data/elasticity/dataset.npz",
    "friction":   "/content/data/friction/dataset.npz",
    "drag":       "/content/data/drag/dataset.npz",
}

with open(OUT_DIR / "spatial_localization.json") as f:
    spatial = json.load(f)
# Temporal is loaded but only used if you want the bottom row; we'll skip
# plotting it given the t=0 confusion from earlier.
try:
    with open(OUT_DIR / "temporal_localization.json") as f:
        temporal = json.load(f)
    HAS_TEMPORAL = True
except FileNotFoundError:
    HAS_TEMPORAL = False


def first_video_last_frame(npz_path):
    video_bytes = 64 * 224 * 224 * 3
    with zipfile.ZipFile(npz_path, "r") as zf:
        with zf.open("frames.npy") as f:
            f.read(6); v = f.read(2)
            hl = int.from_bytes(f.read(4 if v == b"\x02\x00" else 2), "little")
            f.read(hl)
            raw = f.read(video_bytes)
            return np.frombuffer(raw, dtype=np.uint8).reshape(64, 224, 224, 3)[-1]


bg_frames = {prop: first_video_last_frame(PROPERTY_NPZ_PATHS[prop])
             for prop in PROPERTIES_LIST}

# Single row of heatmaps only (skipping the temporal panel given the readout-
# vs-causality issue with Cell E results).
fig, axes = plt.subplots(1, 3, figsize=(13.5, 5.0), gridspec_kw={"wspace": 0.25})

all_grids = [np.array(spatial[p]["r2_grid"]) for p in PROPERTIES_LIST]
vmax = max(g.max() for g in all_grids)
vmin = 0.0

for col, prop in enumerate(PROPERTIES_LIST):
    ax = axes[col]
    bg = bg_frames[prop]
    grid = np.array(spatial[prop]["r2_grid"]).clip(vmin, vmax)

    ax.imshow(bg, extent=[0, 24, 24, 0], aspect="equal")
    im = ax.imshow(grid, cmap="hot", alpha=0.6, extent=[0, 24, 24, 0],
                   vmin=vmin, vmax=vmax, aspect="equal")
    ax.set_title(f"{prop.capitalize()} (L{PEAK_LAYERS_1IDX[prop]}) — spatial R²",
                 fontsize=12)
    ax.set_xticks([0, 12, 24]); ax.set_yticks([0, 12, 24])
    ax.set_xlabel("patch col (w)")
    if col == 0:
        ax.set_ylabel("patch row (h)")

    h_max, w_max = np.unravel_index(np.array(spatial[prop]["r2_grid"]).argmax(),
                                     (24, 24))
    ax.scatter([w_max + 0.5], [h_max + 0.5], s=120, marker="x", color="cyan",
               linewidth=2.8, label=f"max R² = {grid.max():.2f}\n(h={h_max}, w={w_max})")
    ax.legend(loc="upper right", fontsize=9, framealpha=0.85)

cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.8, pad=0.02)
cbar.set_label("Spatial probe R²", fontsize=10)

fig.suptitle("V-JEPA 2 spatial localization of material properties (peak layer per property)",
             fontsize=13, y=1.02)

fig_path = OUT_DIR / "spatial_localization_overlay.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved figure → {fig_path}")

In [ ]:
# === Generate all 4 figures for paper ===
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

OUT_DIR = Path("./probing_results6.0")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

PROPERTIES = ["elasticity", "friction", "drag"]
PEAK_LAYERS_1IDX = {"elasticity": 8, "friction": 12, "drag": 11}


# ============================================================
# Figure 1: Dataset schematic
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))

ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.set_aspect('equal')
ax.add_patch(mpatches.Rectangle((0.4, 0.4), 9.2, 9.2, linewidth=3,
             edgecolor='black', facecolor='#f5f5f5'))
ax.add_patch(mpatches.Circle((4.5, 6), 0.6, color='#1f77b4'))
ax.annotate('', xy=(6.5, 4.2), xytext=(4.8, 5.6),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.set_title("Elasticity\nzero-gravity, 4 walls", fontsize=11)
ax.set_xticks([]); ax.set_yticks([])

ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.set_aspect('equal')
ax.add_patch(mpatches.Rectangle((0.4, 0.4), 9.2, 9.2, linewidth=1.5,
             edgecolor='lightgray', facecolor='white', linestyle='--'))
ax.add_patch(mpatches.Rectangle((0.4, 0.4), 9.2, 1.2, linewidth=2,
             edgecolor='black', facecolor='#888888'))
ax.add_patch(mpatches.Circle((3.5, 2.4), 0.6, color='#d62728'))
ax.annotate('', xy=(5.0, 2.4), xytext=(3.9, 2.4),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.annotate('g', xy=(9.0, 8.2), fontsize=12, style='italic')
ax.annotate('', xy=(9.0, 7.5), xytext=(9.0, 8.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.2))
ax.set_title("Friction\nfloor, linear damping", fontsize=11)
ax.set_xticks([]); ax.set_yticks([])

ax = axes[2]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.set_aspect('equal')
ax.add_patch(mpatches.Rectangle((0.4, 0.4), 9.2, 9.2, linewidth=1.5,
             edgecolor='lightgray', facecolor='white', linestyle='--'))
ax.add_patch(mpatches.Circle((5, 5), 0.6, color='#2ca02c'))
ax.annotate('', xy=(7.5, 7.0), xytext=(5.4, 5.4),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.set_title("Drag\nopen space, exp. damping", fontsize=11)
ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_scenarios.pdf", bbox_inches='tight')
plt.show()
print("Figure 1 saved")


# ============================================================
# Figure 2: Per-layer R² curves
# ============================================================
with open(OUT_DIR / "probe_results.json") as f:
    main_results = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
layers = list(range(1, 13))

for ax, prop in zip(axes, PROPERTIES):
    vjepa = main_results[prop]["vjepa_r2"]
    clip = main_results[prop]["clip_r2"]
    dino = main_results[prop]["dino_r2"]
    ax.plot(layers, vjepa, 'o-', label='V-JEPA 2', linewidth=2.5, color='#d62728')
    ax.plot(layers, clip, 's--', label='CLIP', linewidth=1.5, color='#1f77b4', alpha=0.8)
    ax.plot(layers, dino, '^--', label='DINOv2', linewidth=1.5, color='#2ca02c', alpha=0.8)
    ax.axvline(PEAK_LAYERS_1IDX[prop], color='gray', linestyle=':', linewidth=1)
    ax.set_xlabel("Layer")
    ax.set_title(prop.capitalize())
    ax.set_xticks([1, 4, 8, 12])
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.05, 1.0)

axes[0].set_ylabel("$R^2$")
axes[0].legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_perlayer.pdf", bbox_inches='tight')
plt.show()
print("Figure 2 saved")


# ============================================================
# Figure 3: Factorization heatmap
# ============================================================
with open(OUT_DIR / "transfer_and_extrapolation.json") as f:
    transfer = json.load(f)["cross_property_transfer"]

matrix = np.zeros((3, 3))
layer_keys = ["L8", "L11", "L12"]
for i, src in enumerate(PROPERTIES):
    for j, tgt in enumerate(PROPERTIES):
        if src == tgt:
            matrix[i, j] = 1.00
        else:
            vals = []
            for L in layer_keys:
                key = f"{src}_to_{tgt}"
                entry = transfer[L][key]
                if isinstance(entry, dict):
                    vals.append(abs(entry["pearson_r"]))
                else:
                    vals.append(abs(entry))
            matrix[i, j] = float(np.mean(vals))

fig, ax = plt.subplots(figsize=(5.2, 4.5))
im = ax.imshow(matrix, cmap="Reds", vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(PROPERTIES); ax.set_yticklabels(PROPERTIES)
ax.set_xlabel("Target property")
ax.set_ylabel("Source property")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{matrix[i,j]:.2f}", ha='center', va='center',
                color='white' if matrix[i,j] > 0.5 else 'black', fontsize=11)
plt.colorbar(im, ax=ax, label="|Pearson r|")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_factorization.pdf", bbox_inches='tight')
plt.show()
print("Figure 3 saved")


# ============================================================
# Figure 4: Spatial localization
# ============================================================
with open(OUT_DIR / "spatial_localization.json") as f:
    spatial = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
all_grids = [np.array(spatial[p]["r2_grid"]) for p in PROPERTIES]
vmax = max(g.max() for g in all_grids)

for ax, prop in zip(axes, PROPERTIES):
    grid = np.array(spatial[prop]["r2_grid"]).clip(0, vmax)
    im = ax.imshow(grid, cmap="hot", vmin=0, vmax=vmax, aspect="equal")
    ax.set_title(f"{prop.capitalize()} (L{PEAK_LAYERS_1IDX[prop]})", fontsize=12)
    ax.set_xlabel("patch column (w)")
    if prop == "elasticity":
        ax.set_ylabel("patch row (h)")
    ax.set_xticks([0, 6, 12, 18, 23])
    ax.set_yticks([0, 6, 12, 18, 23])
    h_max, w_max = np.unravel_index(grid.argmax(), grid.shape)
    ax.scatter([w_max], [h_max], s=110, marker="x", color="cyan",
               linewidth=2.8, label=f"max R² = {grid.max():.2f}\n(h={h_max}, w={w_max})")
    ax.legend(loc="upper right", fontsize=9, framealpha=0.85)

cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.8, pad=0.02)
cbar.set_label("Spatial probe $R^2$")
plt.savefig(FIG_DIR / "fig4_spatial.pdf", bbox_inches='tight')
plt.show()
print("Figure 4 saved")

print("\nAll 4 figures generated.")